# GobyLLM — 25M Parameter LLM with Learned Early Exit

A novel, compact language model trained on Stanford Alpaca + tool-calling data.
**First <50M model with trained early exit** — easy queries skip 70% of layers.

**Architecture:**
- 25.5M parameters, 10-layer transformer
- **Learned Early Exit** — per-layer routers decide when to stop (0.02% overhead)
- Parallel Residual Blocks (PaLM-style) — attention & FFN in parallel
- Grouped Query Attention — 8 query heads, 2 KV heads (4x less KV cache)
- Auxiliary exit losses at every layer — model learns to decode from any depth
- SwiGLU FFN, RMSNorm, RoPE, BPE tokenizer (8192 vocab)

**Capabilities:** General instruction following, reasoning, OpenAI-compatible tool calling

**Edge performance:** Simple commands exit at layer 2-3 → **~3x faster** on RPi

## 1. Setup

In [ ]:
!pip install -q torch tokenizers tqdm requests numpy

import torch
print(f'PyTorch {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
import os
os.makedirs('/content/goby', exist_ok=True)
os.chdir('/content/goby')
print(f'Working dir: {os.getcwd()}')

## 2. Source Files

### `config.py`

In [ ]:
%%writefile config.py
"""GobyLLM configuration."""

from dataclasses import dataclass


@dataclass
class GobyConfig:
    # ── Architecture ────────────────────────────────────────────────────
    vocab_size: int = 8192
    max_seq_len: int = 512
    d_model: int = 512
    n_layers: int = 10
    n_heads: int = 8          # query heads
    n_kv_heads: int = 2       # GQA key-value heads
    ffn_hidden: int = 960     # SwiGLU intermediate dim
    dropout: float = 0.1
    rope_base: float = 10000.0
    norm_eps: float = 1e-5
    parallel_residual: bool = True  # PaLM-style parallel attn + FFN

    # ── Early exit ──────────────────────────────────────────────────────
    early_exit: bool = True
    min_exit_layer: int = 2        # don't exit before this layer
    exit_threshold: float = 0.8    # router confidence to exit during inference
    exit_loss_weight: float = 0.1  # weight for auxiliary exit losses
    router_loss_weight: float = 0.02  # weight for router BCE loss

    # ── Special tokens ──────────────────────────────────────────────────
    pad_id: int = 0
    bos_id: int = 1           # <|im_start|>
    eos_id: int = 2           # <|im_end|>


@dataclass
class TrainConfig:
    batch_size: int = 32
    learning_rate: float = 3e-4
    min_lr: float = 3e-5
    weight_decay: float = 0.1
    warmup_steps: int = 300
    max_steps: int = 10000
    eval_interval: int = 400
    save_interval: int = 1000
    grad_clip: float = 1.0
    device: str = "auto"
    seed: int = 42
    data_dir: str = "data"
    output_dir: str = "checkpoints"


### `model.py`

In [ ]:
%%writefile model.py
"""
GobyLLM — 25M parameter LLM with learned early exit for edge devices.

Architecture:
1. Parallel Residual Blocks (PaLM-style) — attn + FFN in parallel
2. Grouped Query Attention — 8 query / 2 KV heads (4x less KV cache)
3. Learned Early Exit — each layer has a tiny router (512→1) that decides
   whether to stop. Easy queries ("turn on lights") exit at layer 2-3,
   hard queries use all 10 layers. First <50M model with this design.
4. SwiGLU FFN, RMSNorm, RoPE, weight-tied embeddings

Training: auxiliary LM loss at every layer teaches the model to produce
decodable representations early. Router heads are trained to predict when
an early exit would give the same answer as the full model.

Inference on RPi: easy queries run 3x faster by skipping 70% of layers.
"""

import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from config import GobyConfig


class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-5):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(dim))
        self.eps = eps

    def forward(self, x):
        norm = x.float().pow(2).mean(-1, keepdim=True).add(self.eps).rsqrt()
        return (x.float() * norm).type_as(x) * self.weight


def precompute_rope(dim, max_seq_len, base=10000.0):
    freqs = 1.0 / (base ** (torch.arange(0, dim, 2).float() / dim))
    t = torch.arange(max_seq_len, dtype=torch.float32)
    freqs = torch.outer(t, freqs)
    return freqs.cos(), freqs.sin()


def apply_rope(x, cos, sin):
    seq_len = x.shape[2]
    cos = cos[:seq_len].unsqueeze(0).unsqueeze(0)
    sin = sin[:seq_len].unsqueeze(0).unsqueeze(0)
    x1, x2 = x[..., :x.shape[-1] // 2], x[..., x.shape[-1] // 2:]
    return torch.cat([x1 * cos - x2 * sin, x2 * cos + x1 * sin], dim=-1)


class GroupedQueryAttention(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.n_heads = config.n_heads
        self.n_kv_heads = config.n_kv_heads
        self.head_dim = config.d_model // config.n_heads
        self.n_rep = self.n_heads // self.n_kv_heads

        self.wq = nn.Linear(config.d_model, config.n_heads * self.head_dim, bias=False)
        self.wk = nn.Linear(config.d_model, config.n_kv_heads * self.head_dim, bias=False)
        self.wv = nn.Linear(config.d_model, config.n_kv_heads * self.head_dim, bias=False)
        self.wo = nn.Linear(config.n_heads * self.head_dim, config.d_model, bias=False)
        self.attn_dropout = nn.Dropout(config.dropout)

    def forward(self, x, cos, sin, mask=None):
        B, T, _ = x.shape
        q = self.wq(x).view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        k = self.wk(x).view(B, T, self.n_kv_heads, self.head_dim).transpose(1, 2)
        v = self.wv(x).view(B, T, self.n_kv_heads, self.head_dim).transpose(1, 2)

        q, k = apply_rope(q, cos, sin), apply_rope(k, cos, sin)

        if self.n_rep > 1:
            k = k.repeat_interleave(self.n_rep, dim=1)
            v = v.repeat_interleave(self.n_rep, dim=1)

        attn = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_dim)
        if mask is not None:
            attn = attn.masked_fill(mask == 0, float("-inf"))
        attn = self.attn_dropout(F.softmax(attn, dim=-1))
        return self.wo((attn @ v).transpose(1, 2).contiguous().view(B, T, -1))


class SwiGLUFFN(nn.Module):
    def __init__(self, config):
        super().__init__()
        h = config.ffn_hidden
        self.w_gate = nn.Linear(config.d_model, h, bias=False)
        self.w_up = nn.Linear(config.d_model, h, bias=False)
        self.w_down = nn.Linear(h, config.d_model, bias=False)
        self.dropout = nn.Dropout(config.dropout)

    def forward(self, x):
        return self.dropout(self.w_down(F.silu(self.w_gate(x)) * self.w_up(x)))


class GobyBlock(nn.Module):
    """Parallel residual: x = x + attn(norm(x)) + ffn(norm(x))"""
    def __init__(self, config):
        super().__init__()
        self.parallel = config.parallel_residual
        self.norm = RMSNorm(config.d_model, config.norm_eps)
        self.attn = GroupedQueryAttention(config)
        self.ffn = SwiGLUFFN(config)
        if not self.parallel:
            self.norm2 = RMSNorm(config.d_model, config.norm_eps)

    def forward(self, x, cos, sin, mask=None):
        if self.parallel:
            h = self.norm(x)
            return x + self.attn(h, cos, sin, mask) + self.ffn(h)
        else:
            x = x + self.attn(self.norm(x), cos, sin, mask)
            return x + self.ffn(self.norm2(x))


class GobyLLM(nn.Module):
    def __init__(self, config: GobyConfig):
        super().__init__()
        self.config = config

        self.tok_emb = nn.Embedding(config.vocab_size, config.d_model)
        self.drop = nn.Dropout(config.dropout)
        self.blocks = nn.ModuleList([GobyBlock(config) for _ in range(config.n_layers)])
        self.norm = RMSNorm(config.d_model, config.norm_eps)
        self.lm_head = nn.Linear(config.d_model, config.vocab_size, bias=False)
        self.lm_head.weight = self.tok_emb.weight  # tie weights

        # Early exit routers: tiny linear head per layer (d_model → 1)
        if config.early_exit:
            self.exit_routers = nn.ModuleList([
                nn.Linear(config.d_model, 1) for _ in range(config.n_layers)
            ])

        # RoPE
        head_dim = config.d_model // config.n_heads
        cos, sin = precompute_rope(head_dim, config.max_seq_len, config.rope_base)
        self.register_buffer("rope_cos", cos)
        self.register_buffer("rope_sin", sin)

        self.apply(self._init_weights)

    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)
            if m.bias is not None:
                nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Embedding):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None):
        """
        Training forward: runs ALL layers, computes exit losses at each layer
        to teach the model to produce decodable outputs early. Routers learn
        to predict when early exit matches the final output.
        """
        B, T = idx.shape
        V = self.config.vocab_size
        x = self.drop(self.tok_emb(idx))
        mask = torch.tril(torch.ones(T, T, device=idx.device)).unsqueeze(0).unsqueeze(0)

        exit_losses = []
        exit_preds = []   # argmax at each layer (no grad, for router targets)
        router_logits = []

        for i, block in enumerate(self.blocks):
            x = block(x, self.rope_cos, self.rope_sin, mask)

            if targets is not None and self.config.early_exit:
                # Auxiliary LM loss at this layer
                exit_logits_i = self.lm_head(self.norm(x))
                el = F.cross_entropy(
                    exit_logits_i.view(-1, V), targets.view(-1), ignore_index=0
                )
                exit_losses.append(el)

                # Store argmax predictions for router training (no grad)
                with torch.no_grad():
                    exit_preds.append(exit_logits_i.argmax(dim=-1))

                # Router confidence (detached input so router doesn't affect hidden path)
                rp = self.exit_routers[i](x.detach().mean(dim=1))  # [B, 1]
                router_logits.append(rp)

        # Final output
        logits = self.lm_head(self.norm(x))

        loss = None
        if targets is not None:
            final_loss = F.cross_entropy(
                logits.view(-1, V), targets.view(-1), ignore_index=0
            )

            if self.config.early_exit and exit_losses:
                # Weighted auxiliary loss (later layers contribute more)
                n = len(exit_losses)
                weights = [(i + 1) / n for i in range(n)]
                w_sum = sum(weights)
                aux_loss = sum(w * l for w, l in zip(weights, exit_losses)) / w_sum

                # Router targets: agreement between layer-i predictions and final
                with torch.no_grad():
                    final_preds = logits.argmax(dim=-1)  # [B, T]

                router_loss = torch.tensor(0.0, device=idx.device)
                for i, rp in enumerate(router_logits):
                    agreement = (exit_preds[i] == final_preds).float()
                    target = agreement.mean(dim=-1, keepdim=True)  # [B, 1]
                    router_loss = router_loss + F.binary_cross_entropy_with_logits(rp, target)
                router_loss = router_loss / len(router_logits)

                loss = (
                    final_loss
                    + self.config.exit_loss_weight * aux_loss
                    + self.config.router_loss_weight * router_loss
                )
            else:
                loss = final_loss

        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens=256, temperature=0.7, top_k=50,
                 exit_threshold=None):
        """
        Autoregressive generation with early exit.
        Returns (token_ids, list_of_exit_layers).
        """
        self.eval()
        threshold = exit_threshold if exit_threshold is not None else self.config.exit_threshold
        use_ee = self.config.early_exit and hasattr(self, "exit_routers")
        exit_layers = []

        for _ in range(max_new_tokens):
            idx_cond = idx[:, -self.config.max_seq_len:]
            B, T = idx_cond.shape

            x = self.tok_emb(idx_cond)
            mask = torch.tril(torch.ones(T, T, device=x.device)).unsqueeze(0).unsqueeze(0)

            exited_at = self.config.n_layers
            for i, block in enumerate(self.blocks):
                x = block(x, self.rope_cos, self.rope_sin, mask)

                # Check early exit (skip first few layers and last layer)
                if use_ee and i >= self.config.min_exit_layer and i < self.config.n_layers - 1:
                    conf = torch.sigmoid(self.exit_routers[i](x[:, -1:, :].mean(dim=1)))
                    if conf.item() > threshold:
                        exited_at = i + 1
                        break

            exit_layers.append(exited_at)

            logits = self.lm_head(self.norm(x))
            logits = logits[:, -1, :] / temperature
            if top_k > 0:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = float("-inf")
            probs = F.softmax(logits, dim=-1)
            next_id = torch.multinomial(probs, num_samples=1)
            idx = torch.cat([idx, next_id], dim=1)

            if next_id.item() == self.config.eos_id:
                break

        return idx, exit_layers

    def param_count(self):
        total = sum(p.numel() for p in self.parameters())
        router = sum(p.numel() for p in self.exit_routers.parameters()) if self.config.early_exit else 0
        return total, router

    def param_summary(self):
        total, router = self.param_count()
        core = total - router
        return (
            f"GobyLLM: {total:,} total params ({total/1e6:.2f}M)\n"
            f"  Core model:  {core:,} ({core/1e6:.2f}M)\n"
            f"  Exit routers: {router:,} ({router/1e6:.4f}M) — {router/total*100:.2f}% overhead"
        )


if __name__ == "__main__":
    config = GobyConfig()
    model = GobyLLM(config)
    print(model.param_summary())
    print(f"\n  d_model={config.d_model}, n_layers={config.n_layers}")
    print(f"  GQA: {config.n_heads}Q / {config.n_kv_heads}KV")
    print(f"  Parallel residual: {config.parallel_residual}")
    print(f"  Early exit: min_layer={config.min_exit_layer}, threshold={config.exit_threshold}")

    x = torch.randint(0, config.vocab_size, (2, 64))
    logits, _ = model(x)
    print(f"\n  Forward: {x.shape} -> {logits.shape}")

    # Test generation with early exit
    prompt = torch.randint(0, config.vocab_size, (1, 8))
    out, exits = model.generate(prompt, max_new_tokens=16)
    avg_exit = sum(exits) / len(exits)
    print(f"  Generate: {prompt.shape[1]} -> {out.shape[1]} tokens")
    print(f"  Exit layers: {exits}")
    print(f"  Avg exit layer: {avg_exit:.1f} / {config.n_layers}")


### `generate_data.py`

In [ ]:
%%writefile generate_data.py
"""
Generate training data for EdgeSense LLM.

Every sample is procedurally generated — no fixed example lists.
Randomizes: entities, values, phrasing, thinking style, response style,
tool combinations, system prompts.

Skills trained:
1. Natural language → tool call (any domain)
2. Read arbitrary tool schemas
3. Step-by-step reasoning
4. Conversational ability
5. Knowing when NOT to act
"""

import json
import random
import os
from collections import Counter

random.seed(42)


# ══════════════════════════════════════════════════════════════════════════════
#  BUILDING BLOCKS — combinatorial parts for generating unique samples
# ══════════════════════════════════════════════════════════════════════════════

def T(name, desc, params):
    return {
        "type": "function",
        "function": {
            "name": name,
            "description": desc,
            "parameters": {
                "type": "object",
                "properties": params,
                "required": list(params.keys()),
            },
        },
    }


def pick(lst):
    return random.choice(lst)


def pick_n(lst, n):
    return random.sample(lst, min(n, len(lst)))


# ── Entities ────────────────────────────────────────────────────────────────

ROOMS = ["living room", "bedroom", "kitchen", "bathroom", "office", "hallway",
         "garage", "basement", "attic", "dining room", "nursery", "guest room",
         "master bedroom", "laundry room", "patio", "porch", "den", "study"]

ZONES = ["zone A", "zone B", "zone C", "zone D", "north wing", "south wing",
         "east wing", "west wing", "building 1", "building 2", "floor 1",
         "floor 2", "floor 3", "main hall", "reception", "loading dock"]

INDUSTRIAL_LOCATIONS = ["server room", "warehouse", "lab", "clean room",
                        "control room", "workshop", "storage", "boiler room",
                        "compressor room", "electrical room", "rooftop",
                        "greenhouse", "cold storage", "paint booth"]

DEVICES = {
    "lights": ["lights", "lamp", "light strip", "chandelier", "overhead light", "desk lamp", "floor lamp"],
    "fans": ["fan", "ceiling fan", "exhaust fan", "desk fan", "tower fan"],
    "heaters": ["heater", "space heater", "radiator", "heat lamp", "floor heater"],
    "coolers": ["AC", "air conditioner", "cooler", "mini split", "window unit"],
    "speakers": ["speaker", "smart speaker", "soundbar", "bedroom speaker", "kitchen speaker"],
    "doors": ["front door", "back door", "garage door", "side door", "gate"],
    "blinds": ["blinds", "curtains", "shades", "shutters"],
    "appliances": ["coffee maker", "oven", "dishwasher", "washing machine", "dryer", "robot vacuum", "dehumidifier", "humidifier", "air purifier"],
}

SENSOR_TYPES = {
    "temperature": {"unit": "°C", "units_alt": ["celsius", "degrees", "°C"], "normal": (18, 26), "warn": (10, 35), "crit": (-5, 50)},
    "humidity": {"unit": "%", "units_alt": ["percent", "%", "% humidity"], "normal": (30, 60), "warn": (15, 80), "crit": (5, 95)},
    "co2": {"unit": "ppm", "units_alt": ["ppm", "parts per million"], "normal": (350, 800), "warn": (800, 1500), "crit": (1500, 5000)},
    "noise": {"unit": "dB", "units_alt": ["dB", "decibels"], "normal": (20, 55), "warn": (55, 80), "crit": (80, 120)},
    "light_level": {"unit": "lux", "units_alt": ["lux"], "normal": (200, 500), "warn": (30, 800), "crit": (0, 2000)},
    "pressure": {"unit": "hPa", "units_alt": ["hPa", "millibars", "mbar"], "normal": (1000, 1025), "warn": (980, 1040), "crit": (950, 1060)},
    "gas": {"unit": "ppm", "units_alt": ["ppm"], "normal": (0, 10), "warn": (10, 50), "crit": (50, 500)},
    "vibration": {"unit": "mm/s", "units_alt": ["mm/s"], "normal": (0, 2.5), "warn": (2.5, 7), "crit": (7, 30)},
    "voltage": {"unit": "V", "units_alt": ["volts", "V"], "normal": (220, 240), "warn": (200, 250), "crit": (180, 270)},
    "battery": {"unit": "%", "units_alt": ["percent", "%"], "normal": (50, 100), "warn": (15, 50), "crit": (0, 15)},
    "water_level": {"unit": "cm", "units_alt": ["cm", "centimeters"], "normal": (10, 50), "warn": (5, 80), "crit": (0, 120)},
    "soil_moisture": {"unit": "%", "units_alt": ["percent", "%"], "normal": (40, 70), "warn": (15, 85), "crit": (5, 100)},
    "wind_speed": {"unit": "km/h", "units_alt": ["km/h", "kph"], "normal": (0, 25), "warn": (25, 60), "crit": (60, 150)},
    "motion": {"unit": "events/hr", "units_alt": ["events", "events per hour"], "normal": (0, 5), "warn": (5, 20), "crit": (20, 100)},
    "pm25": {"unit": "µg/m³", "units_alt": ["µg/m³", "ug/m3"], "normal": (0, 12), "warn": (12, 35), "crit": (35, 300)},
}

SENSOR_RISKS = {
    "temperature": {"high": "overheating, equipment damage, discomfort", "low": "freezing pipes, hypothermia, equipment malfunction"},
    "humidity": {"high": "mold, condensation, corrosion", "low": "static, dry skin, respiratory irritation"},
    "co2": {"high": "drowsiness, headaches, poor concentration", "low": "unusual, check sensor"},
    "noise": {"high": "hearing damage, stress", "low": "check sensor"},
    "light_level": {"high": "glare, eye strain", "low": "eye strain, safety hazard"},
    "gas": {"high": "explosion risk, poisoning — life safety threat", "low": "not applicable"},
    "vibration": {"high": "bearing failure, equipment damage", "low": "not typical"},
    "voltage": {"high": "equipment damage, fire risk", "low": "brownout, equipment malfunction"},
    "battery": {"high": "not applicable", "low": "device shutdown, data loss"},
    "water_level": {"high": "overflow, flooding", "low": "dry run damage, insufficient supply"},
    "soil_moisture": {"high": "root rot, waterlogging", "low": "plant stress, wilting"},
    "wind_speed": {"high": "structural damage, flying debris", "low": "not typical"},
    "motion": {"high": "security concern, unauthorized access", "low": "sensor may be blocked"},
    "pressure": {"high": "over-pressurization", "low": "vacuum, cavitation"},
    "pm25": {"high": "respiratory issues, poor air quality", "low": "not typical"},
}

PEOPLE = ["John", "Sarah", "Mike", "Emma", "David", "Lisa", "Tom", "Anna",
          "James", "Maria", "Alex", "Chris", "Sam", "Pat", "Jordan", "Taylor"]

TIMES = ["7:00 AM", "7:30 AM", "8:00 AM", "8:15 AM", "9:00 AM", "10:00 AM",
         "11:00 AM", "12:00 PM", "1:00 PM", "2:00 PM", "2:30 PM", "3:00 PM",
         "4:00 PM", "5:00 PM", "6:00 PM", "6:30 PM", "7:00 PM", "8:00 PM",
         "9:00 PM", "10:00 PM", "10:30 PM", "11:00 PM"]

DURATIONS = ["5 minutes", "10 minutes", "15 minutes", "20 minutes", "30 minutes",
             "45 minutes", "1 hour", "2 hours", "3 hours"]

MUSIC_GENRES = ["jazz", "classical", "rock", "pop", "lo-fi", "ambient", "blues",
                "country", "electronic", "hip hop", "R&B", "folk", "indie"]

TASKS_TO_REMEMBER = ["call the plumber", "check the sensors", "water the plants",
                     "pick up groceries", "submit the report", "feed the dog",
                     "take out the trash", "pay the electric bill", "order supplies",
                     "schedule maintenance", "review the logs", "update firmware",
                     "replace batteries", "clean the filters", "test the backup generator"]

PUMP_IDS = [f"pump_{i}" for i in range(1, 13)]
VALVE_IDS = [f"valve_{i}" for i in range(1, 16)]
FAN_IDS = [f"fan_{i}" for i in range(1, 10)] + [f"{z}_fan" for z in ["office", "warehouse", "lab", "server_room", "kitchen"]]
SENSOR_IDS = [f"sensor_{i:03d}" for i in range(1, 80)]

BRIGHTNESS_LEVELS = list(range(5, 100, 5)) + [1, 15, 25, 42, 67, 73, 88]
TEMPERATURES = [t / 2 for t in range(32, 56)]  # 16.0 to 27.5 in 0.5 steps

# ── Phrasing variants ──────────────────────────────────────────────────────

TURN_ON_PHRASES = [
    "Turn on the {device} in the {room}",
    "Can you turn on the {room} {device}?",
    "Switch on the {device} in the {room}",
    "Please turn on the {device} in the {room}",
    "Enable the {room} {device}",
    "I need the {device} on in the {room}",
    "Start the {device} in the {room}",
    "Power on the {room} {device}",
    "Hey, turn the {room} {device} on",
    "Could you switch the {device} on in the {room}?",
]

TURN_OFF_PHRASES = [
    "Turn off the {device} in the {room}",
    "Can you turn off the {room} {device}?",
    "Switch off the {device} in the {room}",
    "Please shut off the {device} in the {room}",
    "Disable the {room} {device}",
    "Kill the {device} in the {room}",
    "I don't need the {device} in the {room} anymore",
    "Shut down the {room} {device}",
    "Turn the {room} {device} off please",
    "Power off the {room} {device}",
]

BRIGHTNESS_PHRASES = [
    "Set the {room} lights to {level}%",
    "Dim the {room} lights to {level} percent",
    "Make the {room} lights {level}%",
    "Set {room} light brightness to {level}",
    "Change the {room} lights to {level}%",
    "I want the {room} lights at {level}%",
    "Put the {room} lights on {level} percent",
    "Adjust {room} lighting to {level}%",
]

TEMP_PHRASES = [
    "Set the temperature to {temp} degrees",
    "Make it {temp} degrees in here",
    "Set thermostat to {temp}",
    "I want it {temp} degrees",
    "Can you set the temperature to {temp}?",
    "Change the temperature to {temp} celsius",
    "Adjust the thermostat to {temp}",
    "Set heating to {temp} degrees",
]

IMPLICIT_HOT_PHRASES = [
    "It's too hot in here",
    "It's really warm in the {room}",
    "I'm sweating, it's boiling in here",
    "Can you cool down the {room}?",
    "The {room} is way too warm",
    "It's stuffy in here",
    "I'm overheating",
    "This room is like an oven",
    "Way too warm in the {room}",
    "I need it cooler in here",
]

IMPLICIT_COLD_PHRASES = [
    "It's freezing in here",
    "It's really cold in the {room}",
    "I'm cold, can you warm it up?",
    "The {room} is too chilly",
    "It's way too cold in here",
    "Brrr, it's cold",
    "Can you heat up the {room}?",
    "I need more warmth in here",
    "This room is an icebox",
    "I'm shivering, turn up the heat",
]

IMPLICIT_DARK_PHRASES = [
    "It's too dark in here",
    "I can't see anything in the {room}",
    "The {room} is really dim",
    "It's dark in the {room}",
    "I need more light in the {room}",
    "Can you brighten up the {room}?",
    "It's gloomy in here",
    "The lighting is terrible in the {room}",
]

IMPLICIT_BRIGHT_PHRASES = [
    "The lights are too bright in the {room}",
    "It's way too bright in here",
    "Can you dim the {room} a bit?",
    "The {room} lights are blinding",
    "Too much light in the {room}",
    "Tone down the lights in the {room}",
]

CONFIRM_RESPONSES = [
    "Done.", "Got it.", "All set.", "Sure thing.",
    "On it.", "You got it.", "No problem.", "Taken care of.",
]

THINK_STARTERS = [
    "The user wants to {action}. ",
    "I need to {action}. ",
    "The request is to {action}. ",
    "Let me {action}. ",
    "I should {action}. ",
]

SYSTEMS = [
    "You are a helpful assistant. Use the provided tools when appropriate.",
    "You are a smart home assistant. Help the user control their home and answer questions.",
    "You are an IoT monitoring assistant. Analyze data and take action when needed.",
    "You are a helpful AI assistant running on an edge device. Be concise and useful.",
    "You are an intelligent assistant. Think step by step, then respond or use tools as needed.",
    "You are a facility management assistant. Monitor conditions and control equipment.",
    "You are a personal assistant. Help with tasks, reminders, and device control.",
    "You are an assistant. Respond to commands, answer questions, and use tools when helpful.",
    "You are a building automation assistant. Control systems and respond to sensor data.",
    "You are a helpful AI. Use your tools to assist the user.",
]


# ── Tool pools ──────────────────────────────────────────────────────────────

SMART_HOME_TOOLS = [
    T("turn_on", "Turn on a device", {"device": {"type": "string", "description": "Device name"}}),
    T("turn_off", "Turn off a device", {"device": {"type": "string", "description": "Device name"}}),
    T("set_temperature", "Set thermostat temperature and mode", {
        "temperature": {"type": "number", "description": "Target temperature in celsius"},
        "mode": {"type": "string", "enum": ["heat", "cool", "auto"], "description": "HVAC mode"},
    }),
    T("set_brightness", "Set light brightness", {
        "device": {"type": "string", "description": "Light name"},
        "level": {"type": "integer", "description": "Brightness 0-100"},
    }),
    T("lock_door", "Lock a door", {"door": {"type": "string", "description": "Door name"}}),
    T("unlock_door", "Unlock a door", {"door": {"type": "string", "description": "Door name"}}),
    T("set_fan_speed", "Set fan speed", {
        "device": {"type": "string", "description": "Fan name"},
        "speed": {"type": "string", "enum": ["off", "low", "medium", "high"]},
    }),
    T("play_music", "Play music", {
        "speaker": {"type": "string"},
        "query": {"type": "string", "description": "Song, artist, or genre"},
    }),
    T("set_alarm", "Set an alarm", {"time": {"type": "string", "description": "Alarm time"}}),
    T("set_timer", "Set a countdown timer", {
        "duration": {"type": "string", "description": "Duration"},
        "label": {"type": "string", "description": "What the timer is for"},
    }),
    T("open_blinds", "Open window blinds/curtains", {"room": {"type": "string"}}),
    T("close_blinds", "Close window blinds/curtains", {"room": {"type": "string"}}),
    T("start_vacuum", "Start robot vacuum", {"room": {"type": "string", "description": "Room or 'all'"}}),
    T("send_notification", "Send a push notification", {
        "message": {"type": "string"},
        "priority": {"type": "string", "enum": ["low", "normal", "high", "urgent"]},
    }),
]

INDUSTRIAL_TOOLS = [
    T("send_alert", "Send monitoring alert", {
        "level": {"type": "string", "enum": ["info", "warning", "critical", "emergency"]},
        "message": {"type": "string"},
        "zone": {"type": "string"},
    }),
    T("start_pump", "Start a pump", {
        "pump_id": {"type": "string"},
        "flow_rate": {"type": "string", "enum": ["low", "medium", "high", "max"]},
    }),
    T("stop_pump", "Stop a pump", {"pump_id": {"type": "string"}}),
    T("open_valve", "Open a valve", {"valve_id": {"type": "string"}}),
    T("close_valve", "Close a valve", {"valve_id": {"type": "string"}}),
    T("enable_fan", "Enable ventilation fan", {
        "fan_id": {"type": "string"},
        "speed": {"type": "string", "enum": ["low", "medium", "high"]},
    }),
    T("request_maintenance", "Create maintenance work order", {
        "device_id": {"type": "string"},
        "issue": {"type": "string"},
        "priority": {"type": "string", "enum": ["low", "medium", "high"]},
    }),
    T("evacuate_zone", "Issue evacuation alert", {
        "zone": {"type": "string"},
        "reason": {"type": "string"},
    }),
    T("log_event", "Record event to system log", {
        "event": {"type": "string"},
        "severity": {"type": "string", "enum": ["info", "warning", "error", "critical"]},
    }),
    T("calibrate_sensor", "Start sensor calibration", {
        "sensor_id": {"type": "string"},
        "reason": {"type": "string"},
    }),
]

GENERAL_TOOLS = [
    T("search", "Search for information", {"query": {"type": "string"}}),
    T("get_weather", "Get weather for a location", {"location": {"type": "string"}}),
    T("calculate", "Evaluate math expression", {"expression": {"type": "string"}}),
    T("send_message", "Send a message", {
        "to": {"type": "string"},
        "message": {"type": "string"},
    }),
    T("create_reminder", "Create a reminder", {
        "text": {"type": "string"},
        "when": {"type": "string", "description": "When to remind"},
    }),
    T("take_note", "Save a note", {
        "title": {"type": "string"},
        "content": {"type": "string"},
    }),
    T("convert_units", "Convert between units", {
        "value": {"type": "number"},
        "from_unit": {"type": "string"},
        "to_unit": {"type": "string"},
    }),
]

AGRICULTURE_TOOLS = [
    T("water_plants", "Water a garden zone", {
        "zone": {"type": "string"},
        "duration": {"type": "string"},
    }),
    T("adjust_greenhouse", "Adjust greenhouse settings", {
        "parameter": {"type": "string", "enum": ["temperature", "humidity", "ventilation", "shade"]},
        "value": {"type": "string"},
    }),
    T("feed_animals", "Dispense animal feed", {
        "feeder": {"type": "string"},
        "amount": {"type": "string", "enum": ["small", "normal", "large"]},
    }),
]

ALL_TOOL_POOLS = [SMART_HOME_TOOLS, INDUSTRIAL_TOOLS, GENERAL_TOOLS, AGRICULTURE_TOOLS]


def pick_tools(must_include_names=None, pool=None, n=None):
    if pool is None:
        pool = pick(ALL_TOOL_POOLS + [SMART_HOME_TOOLS + GENERAL_TOOLS, INDUSTRIAL_TOOLS + GENERAL_TOOLS])
    if n is None:
        n = random.randint(3, 7)
    pool = list(pool)
    selected = []
    if must_include_names:
        for name in must_include_names:
            for t in pool:
                if t["function"]["name"] == name:
                    selected.append(t)
                    pool.remove(t)
                    break
    remaining = max(0, n - len(selected))
    if remaining > 0 and pool:
        selected.extend(random.sample(pool, min(remaining, len(pool))))
    random.shuffle(selected)
    return selected


# ══════════════════════════════════════════════════════════════════════════════
#  FORMAT
# ══════════════════════════════════════════════════════════════════════════════

def format_sample(s):
    parts = []
    sys_content = s["system"]
    if s.get("tools"):
        sys_content += f"\n\n# Tools\n{json.dumps(s['tools'], separators=(',', ':'))}"
    parts.append(f"<|im_start|>system\n{sys_content}<|im_end|>")
    for m in s.get("history", []):
        parts.append(f"<|im_start|>{m['role']}\n{m['content']}<|im_end|>")
    parts.append(f"<|im_start|>user\n{s['input']}<|im_end|>")
    a = ""
    if s.get("think"):
        a += f"<think>{s['think']}</think>\n"
    if s.get("tool_call"):
        tc = s["tool_call"]
        a += f"<tool_call>{json.dumps({'name': tc['name'], 'arguments': tc['arguments']}, separators=(',', ':'))}</tool_call>\n"
    a += s["response"]
    parts.append(f"<|im_start|>assistant\n{a}<|im_end|>")
    return "\n".join(parts)


def to_openai(s):
    msgs = [{"role": "system", "content": s["system"]}]
    for m in s.get("history", []):
        msgs.append(m)
    msgs.append({"role": "user", "content": s["input"]})
    amsg = {"role": "assistant", "content": s["response"]}
    if s.get("tool_call"):
        amsg["tool_calls"] = [{"id": f"call_{random.randint(100000,999999)}", "type": "function", "function": s["tool_call"]}]
    msgs.append(amsg)
    row = {"messages": msgs}
    if s.get("tools"):
        row["tools"] = s["tools"]
    return row


# ══════════════════════════════════════════════════════════════════════════════
#  GENERATORS — each call produces a UNIQUE sample
# ══════════════════════════════════════════════════════════════════════════════


def gen_turn_on():
    room = pick(ROOMS)
    dev_type = pick(list(DEVICES.keys()))
    device = pick(DEVICES[dev_type])
    phrase = pick(TURN_ON_PHRASES).format(device=device, room=room)
    full_name = f"{room} {device}"
    confirm = pick(CONFIRM_RESPONSES)

    return {
        "system": pick(SYSTEMS),
        "tools": pick_tools(must_include_names=["turn_on"], pool=SMART_HOME_TOOLS),
        "input": phrase,
        "think": pick(THINK_STARTERS).format(action=f"turn on the {device} in the {room}") + "I have the turn_on tool for this.",
        "response": f"{confirm} {full_name.capitalize()} is now on.",
        "tool_call": {"name": "turn_on", "arguments": {"device": full_name}},
        "category": "natural_command",
    }


def gen_turn_off():
    room = pick(ROOMS)
    dev_type = pick(list(DEVICES.keys()))
    device = pick(DEVICES[dev_type])
    phrase = pick(TURN_OFF_PHRASES).format(device=device, room=room)
    full_name = f"{room} {device}"
    confirm = pick(CONFIRM_RESPONSES)

    return {
        "system": pick(SYSTEMS),
        "tools": pick_tools(must_include_names=["turn_off"], pool=SMART_HOME_TOOLS),
        "input": phrase,
        "think": pick(THINK_STARTERS).format(action=f"turn off the {device} in the {room}") + "I'll use the turn_off tool.",
        "response": f"{confirm} {full_name.capitalize()} is off.",
        "tool_call": {"name": "turn_off", "arguments": {"device": full_name}},
        "category": "natural_command",
    }


def gen_set_brightness():
    room = pick(ROOMS)
    level = pick(BRIGHTNESS_LEVELS)
    phrase = pick(BRIGHTNESS_PHRASES).format(room=room, level=level)

    return {
        "system": pick(SYSTEMS),
        "tools": pick_tools(must_include_names=["set_brightness"], pool=SMART_HOME_TOOLS),
        "input": phrase,
        "think": f"The user wants the {room} lights at {level}%. I'll use set_brightness.",
        "response": f"{room.capitalize()} lights set to {level}%.",
        "tool_call": {"name": "set_brightness", "arguments": {"device": f"{room} lights", "level": level}},
        "category": "natural_command",
    }


def gen_set_temp():
    temp = pick(TEMPERATURES)
    mode = pick(["heat", "cool", "auto"])
    phrase = pick(TEMP_PHRASES).format(temp=temp)
    if random.random() < 0.3:
        mode_word = {"heat": "heating", "cool": "cooling", "auto": "auto"}[mode]
        phrase += f", {mode_word} mode"

    return {
        "system": pick(SYSTEMS),
        "tools": pick_tools(must_include_names=["set_temperature"], pool=SMART_HOME_TOOLS),
        "input": phrase,
        "think": f"Setting thermostat to {temp}°C, {mode} mode.",
        "response": f"Thermostat set to {temp}°C in {mode} mode.",
        "tool_call": {"name": "set_temperature", "arguments": {"temperature": temp, "mode": mode}},
        "category": "natural_command",
    }


def gen_implicit_hot():
    room = pick(ROOMS)
    phrase = pick(IMPLICIT_HOT_PHRASES).format(room=room)
    temp = pick([t for t in TEMPERATURES if t <= 22])

    responses = [
        f"I'll cool it down — setting the AC to {temp}°C.",
        f"Setting the thermostat to cool mode at {temp}°C. Should feel better soon.",
        f"Turning on cooling. Target: {temp}°C.",
        f"I've set cooling to {temp}°C. Give it a few minutes.",
    ]

    return {
        "system": pick(SYSTEMS),
        "tools": pick_tools(must_include_names=["set_temperature"], pool=SMART_HOME_TOOLS),
        "input": phrase,
        "think": f"The user is too hot. I should turn on cooling to a comfortable temperature like {temp}°C.",
        "response": pick(responses),
        "tool_call": {"name": "set_temperature", "arguments": {"temperature": temp, "mode": "cool"}},
        "category": "implicit_command",
    }


def gen_implicit_cold():
    room = pick(ROOMS)
    phrase = pick(IMPLICIT_COLD_PHRASES).format(room=room)
    temp = pick([t for t in TEMPERATURES if t >= 22])

    responses = [
        f"I'll warm it up — setting heating to {temp}°C.",
        f"Turning on the heater. Target: {temp}°C.",
        f"Setting heat mode to {temp}°C. Should warm up soon.",
        f"Heater set to {temp}°C.",
    ]

    return {
        "system": pick(SYSTEMS),
        "tools": pick_tools(must_include_names=["set_temperature"], pool=SMART_HOME_TOOLS),
        "input": phrase,
        "think": f"The user is cold. I'll set heating to {temp}°C.",
        "response": pick(responses),
        "tool_call": {"name": "set_temperature", "arguments": {"temperature": temp, "mode": "heat"}},
        "category": "implicit_command",
    }


def gen_implicit_dark():
    room = pick(ROOMS)
    phrase = pick(IMPLICIT_DARK_PHRASES).format(room=room)
    level = random.randint(60, 90)

    return {
        "system": pick(SYSTEMS),
        "tools": pick_tools(must_include_names=["set_brightness"], pool=SMART_HOME_TOOLS),
        "input": phrase,
        "think": f"The user says it's too dark in the {room}. I should increase the light level.",
        "response": f"I've turned up the {room} lights to {level}%.",
        "tool_call": {"name": "set_brightness", "arguments": {"device": f"{room} lights", "level": level}},
        "category": "implicit_command",
    }


def gen_implicit_bright():
    room = pick(ROOMS)
    phrase = pick(IMPLICIT_BRIGHT_PHRASES).format(room=room)
    level = random.randint(20, 45)

    return {
        "system": pick(SYSTEMS),
        "tools": pick_tools(must_include_names=["set_brightness"], pool=SMART_HOME_TOOLS),
        "input": phrase,
        "think": f"Too bright in the {room}. I'll dim the lights.",
        "response": f"Dimmed the {room} lights to {level}%.",
        "tool_call": {"name": "set_brightness", "arguments": {"device": f"{room} lights", "level": level}},
        "category": "implicit_command",
    }


def gen_lock_door():
    door = pick(DEVICES["doors"])
    phrases = [f"Lock the {door}", f"Can you lock the {door}?", f"Please lock the {door}",
               f"Make sure the {door} is locked", f"Secure the {door}"]
    return {
        "system": pick(SYSTEMS),
        "tools": pick_tools(must_include_names=["lock_door"], pool=SMART_HOME_TOOLS),
        "input": pick(phrases),
        "think": f"Lock the {door}.",
        "response": f"{door.capitalize()} is now locked.",
        "tool_call": {"name": "lock_door", "arguments": {"door": door}},
        "category": "natural_command",
    }


def gen_music():
    genre = pick(MUSIC_GENRES)
    room = pick(ROOMS)
    speaker = f"{room} speaker"
    phrases = [f"Play some {genre}", f"Put on {genre} music", f"Play {genre} in the {room}",
               f"I want to hear {genre}", f"Can you play {genre} music?"]
    return {
        "system": pick(SYSTEMS),
        "tools": pick_tools(must_include_names=["play_music"], pool=SMART_HOME_TOOLS),
        "input": pick(phrases),
        "think": f"The user wants {genre} music. Playing on {speaker}.",
        "response": f"Playing {genre} on the {speaker}. Enjoy!",
        "tool_call": {"name": "play_music", "arguments": {"speaker": speaker, "query": genre}},
        "category": "natural_command",
    }


def gen_timer_alarm():
    if random.random() < 0.5:
        # Timer
        dur = pick(DURATIONS)
        label = pick(["pasta", "laundry", "meeting", "break", "oven", "meds", "workout", "tea", "rice", "nap"])
        phrases = [f"Set a timer for {dur} for the {label}", f"{dur} timer for {label}",
                   f"Timer {dur}, {label}", f"Start a {dur} countdown for {label}"]
        return {
            "system": pick(SYSTEMS),
            "tools": pick_tools(must_include_names=["set_timer"], pool=SMART_HOME_TOOLS),
            "input": pick(phrases),
            "think": f"Setting a {dur} timer for {label}.",
            "response": f"{dur.capitalize()} timer set for {label}.",
            "tool_call": {"name": "set_timer", "arguments": {"duration": dur, "label": label}},
            "category": "natural_command",
        }
    else:
        # Alarm
        time = pick(TIMES)
        phrases = [f"Set an alarm for {time}", f"Wake me up at {time}",
                   f"Alarm at {time} please", f"I need an alarm for {time}"]
        return {
            "system": pick(SYSTEMS),
            "tools": pick_tools(must_include_names=["set_alarm"], pool=SMART_HOME_TOOLS),
            "input": pick(phrases),
            "think": f"Setting alarm for {time}.",
            "response": f"Alarm set for {time}.",
            "tool_call": {"name": "set_alarm", "arguments": {"time": time}},
            "category": "natural_command",
        }


def gen_blinds():
    room = pick(ROOMS)
    action = pick(["open", "close"])
    cover = pick(DEVICES["blinds"])
    phrases = [f"{action.capitalize()} the {cover} in the {room}", f"Can you {action} the {room} {cover}?",
               f"Please {action} the {cover} in the {room}", f"I want the {room} {cover} {action}ed"]
    tool_name = f"{action}_blinds"
    return {
        "system": pick(SYSTEMS),
        "tools": pick_tools(must_include_names=[tool_name], pool=SMART_HOME_TOOLS),
        "input": pick(phrases),
        "think": f"{action.capitalize()} the {cover} in the {room}.",
        "response": f"{room.capitalize()} {cover} are now {action + ('d' if action == 'close' else '')}.",
        "tool_call": {"name": tool_name, "arguments": {"room": room}},
        "category": "natural_command",
    }


def gen_vacuum():
    room = pick(ROOMS + ["all rooms", "the whole house", "everywhere"])
    target = "all" if room in ["all rooms", "the whole house", "everywhere"] else room
    phrases = [f"Vacuum the {room}", f"Start the vacuum in the {room}", f"Clean the {room}",
               f"Can you vacuum {room}?", f"Run the robot vacuum in the {room}"]
    return {
        "system": pick(SYSTEMS),
        "tools": pick_tools(must_include_names=["start_vacuum"], pool=SMART_HOME_TOOLS),
        "input": pick(phrases),
        "think": f"Starting vacuum in {target}.",
        "response": f"Robot vacuum is cleaning {target if target != 'all' else 'the whole house'} now.",
        "tool_call": {"name": "start_vacuum", "arguments": {"room": target}},
        "category": "natural_command",
    }


# ── Industrial commands ────────────────────────────────────────────────────

def gen_industrial_command():
    cmd_type = pick(["pump_start", "pump_stop", "valve", "fan", "maintenance", "alert", "calibrate"])
    loc = pick(INDUSTRIAL_LOCATIONS + ZONES)

    if cmd_type == "pump_start":
        pid = pick(PUMP_IDS)
        rate = pick(["low", "medium", "high", "max"])
        phrases = [f"Start {pid} at {rate} flow", f"Turn on {pid}, {rate} rate",
                   f"Fire up {pid} at {rate}", f"Enable {pid} at {rate} flow rate"]
        return {
            "system": pick(SYSTEMS),
            "tools": pick_tools(must_include_names=["start_pump"], pool=INDUSTRIAL_TOOLS),
            "input": pick(phrases),
            "think": f"Starting {pid} at {rate} flow rate.",
            "response": f"{pid.capitalize().replace('_', ' ')} running at {rate} flow.",
            "tool_call": {"name": "start_pump", "arguments": {"pump_id": pid, "flow_rate": rate}},
            "category": "natural_command",
        }

    elif cmd_type == "pump_stop":
        pid = pick(PUMP_IDS)
        phrases = [f"Stop {pid}", f"Shut down {pid}", f"Turn off {pid}", f"Kill {pid}"]
        return {
            "system": pick(SYSTEMS),
            "tools": pick_tools(must_include_names=["stop_pump"], pool=INDUSTRIAL_TOOLS),
            "input": pick(phrases),
            "think": f"Stopping {pid}.",
            "response": f"{pid.capitalize().replace('_', ' ')} stopped.",
            "tool_call": {"name": "stop_pump", "arguments": {"pump_id": pid}},
            "category": "natural_command",
        }

    elif cmd_type == "valve":
        vid = pick(VALVE_IDS)
        action = pick(["open", "close"])
        phrases = [f"{action.capitalize()} {vid}", f"Can you {action} {vid}?",
                   f"I need {vid} {action}ed", f"Please {action} {vid}"]
        return {
            "system": pick(SYSTEMS),
            "tools": pick_tools(must_include_names=[f"{action}_valve"], pool=INDUSTRIAL_TOOLS),
            "input": pick(phrases),
            "think": f"{action.capitalize()} {vid}.",
            "response": f"{vid.capitalize().replace('_', ' ')} is now {action + ('d' if action == 'close' else '')}.",
            "tool_call": {"name": f"{action}_valve", "arguments": {"valve_id": vid}},
            "category": "natural_command",
        }

    elif cmd_type == "fan":
        fid = pick(FAN_IDS)
        speed = pick(["low", "medium", "high"])
        phrases = [f"Turn on {fid} at {speed}", f"Enable {fid}, {speed} speed",
                   f"Start {fid} on {speed}", f"Set {fid} to {speed}"]
        return {
            "system": pick(SYSTEMS),
            "tools": pick_tools(must_include_names=["enable_fan"], pool=INDUSTRIAL_TOOLS),
            "input": pick(phrases),
            "think": f"Enabling {fid} at {speed} speed.",
            "response": f"{fid.capitalize().replace('_', ' ')} running at {speed}.",
            "tool_call": {"name": "enable_fan", "arguments": {"fan_id": fid, "speed": speed}},
            "category": "natural_command",
        }

    elif cmd_type == "maintenance":
        sid = pick(SENSOR_IDS)
        issues = ["making unusual noise", "readings seem off", "intermittent connection",
                  "response time is slow", "physical damage visible", "needs recalibration",
                  "error light is on", "display is blank", "overheating"]
        issue = pick(issues)
        priority = pick(["low", "medium", "high"])
        phrases = [f"The {sid} is {issue}", f"We need maintenance on {sid} — {issue}",
                   f"Something's wrong with {sid}, it's {issue}", f"Can you log a maintenance request for {sid}? It's {issue}"]
        return {
            "system": pick(SYSTEMS),
            "tools": pick_tools(must_include_names=["request_maintenance"], pool=INDUSTRIAL_TOOLS),
            "input": pick(phrases),
            "think": f"{sid} has an issue: {issue}. Submitting a {priority}-priority maintenance request.",
            "response": f"Maintenance request submitted for {sid}: {issue}. Priority: {priority}.",
            "tool_call": {"name": "request_maintenance", "arguments": {"device_id": sid, "issue": issue, "priority": priority}},
            "category": "natural_command",
        }

    elif cmd_type == "alert":
        level = pick(["info", "warning", "critical"])
        situations = [
            f"unusual activity in {loc}", f"equipment malfunction in {loc}",
            f"environmental reading out of range in {loc}", f"scheduled inspection overdue for {loc}",
            f"visitor reported issue in {loc}", f"power fluctuation in {loc}",
        ]
        situation = pick(situations)
        phrases = [f"Send a {level} alert about {situation}", f"Alert: {situation}",
                   f"Flag {situation} as {level}", f"Log a {level} alert for {situation}"]
        return {
            "system": pick(SYSTEMS),
            "tools": pick_tools(must_include_names=["send_alert"], pool=INDUSTRIAL_TOOLS),
            "input": pick(phrases),
            "think": f"Sending {level} alert: {situation}.",
            "response": f"{level.capitalize()} alert sent: {situation}.",
            "tool_call": {"name": "send_alert", "arguments": {"level": level, "message": situation, "zone": loc}},
            "category": "natural_command",
        }

    else:  # calibrate
        sid = pick(SENSOR_IDS)
        reasons = ["routine calibration", "readings seem off", "after maintenance", "quarterly schedule", "drift detected"]
        reason = pick(reasons)
        phrases = [f"Calibrate {sid}", f"Start calibration on {sid} — {reason}",
                   f"I need {sid} recalibrated", f"Run calibration for {sid}"]
        return {
            "system": pick(SYSTEMS),
            "tools": pick_tools(must_include_names=["calibrate_sensor"], pool=INDUSTRIAL_TOOLS),
            "input": pick(phrases),
            "think": f"Starting calibration for {sid}. Reason: {reason}.",
            "response": f"Calibration started for {sid} ({reason}).",
            "tool_call": {"name": "calibrate_sensor", "arguments": {"sensor_id": sid, "reason": reason}},
            "category": "natural_command",
        }


# ── General commands ───────────────────────────────────────────────────────

def gen_general_command():
    cmd_type = pick(["reminder", "message", "weather", "calculate", "convert", "note"])

    if cmd_type == "reminder":
        task = pick(TASKS_TO_REMEMBER)
        when = pick(["in 1 hour", "in 2 hours", "in 30 minutes", "tomorrow morning",
                     "tomorrow at 9am", "tonight", "in 3 hours", "at 5pm"])
        phrases = [f"Remind me to {task} {when}", f"Set a reminder: {task} {when}",
                   f"Don't let me forget to {task} — remind me {when}"]
        return {
            "system": pick(SYSTEMS),
            "tools": pick_tools(must_include_names=["create_reminder"], pool=GENERAL_TOOLS),
            "input": pick(phrases),
            "think": f"Setting reminder: {task}, {when}.",
            "response": f"Reminder set: {task} ({when}).",
            "tool_call": {"name": "create_reminder", "arguments": {"text": task, "when": when}},
            "category": "natural_command",
        }

    elif cmd_type == "message":
        person = pick(PEOPLE)
        messages = [f"I'll be {random.randint(5,45)} minutes late",
                    "On my way", "Can you call me?", "Meeting is postponed",
                    "I'll handle it", "Running behind, start without me",
                    "Pick up some milk please", "Everything's fine here"]
        msg = pick(messages)
        phrases = [f"Send a message to {person}: {msg}", f"Text {person} saying {msg}",
                   f"Tell {person} that {msg}", f"Message {person}: {msg}"]
        return {
            "system": pick(SYSTEMS),
            "tools": pick_tools(must_include_names=["send_message"], pool=GENERAL_TOOLS),
            "input": pick(phrases),
            "think": f"Sending message to {person}: \"{msg}\".",
            "response": f"Message sent to {person}.",
            "tool_call": {"name": "send_message", "arguments": {"to": person, "message": msg}},
            "category": "natural_command",
        }

    elif cmd_type == "weather":
        cities = ["Tokyo", "London", "New York", "Paris", "Sydney", "Berlin",
                  "Toronto", "Mumbai", "Seoul", "Portland", "Seattle", "Austin",
                  "Denver", "Chicago", "Boston", "Miami", "San Francisco"]
        city = pick(cities)
        phrases = [f"What's the weather in {city}?", f"How's the weather in {city}?",
                   f"Weather for {city}", f"Is it raining in {city}?",
                   f"What's it like outside in {city}?"]
        return {
            "system": pick(SYSTEMS),
            "tools": pick_tools(must_include_names=["get_weather"], pool=GENERAL_TOOLS),
            "input": pick(phrases),
            "think": f"The user wants weather for {city}. Using get_weather tool.",
            "response": f"Let me check the weather in {city} for you.",
            "tool_call": {"name": "get_weather", "arguments": {"location": city}},
            "category": "natural_command",
        }

    elif cmd_type == "calculate":
        ops = [
            (f"{random.randint(1,99)} * {random.randint(1,99)}", None),
            (f"{random.randint(100,999)} + {random.randint(100,999)}", None),
            (f"{random.randint(1,50)}% of {random.randint(50,500)}", None),
            (f"{random.randint(10,200)} / {random.randint(2,10)}", None),
            (f"square root of {random.choice([4,9,16,25,36,49,64,81,100,144])}", None),
        ]
        expr_natural, _ = pick(ops)
        expr_math = expr_natural.replace("% of", "/100 *").replace("square root of", "sqrt(")
        if "sqrt(" in expr_math:
            expr_math += ")"
        phrases = [f"What's {expr_natural}?", f"Calculate {expr_natural}",
                   f"How much is {expr_natural}?", f"{expr_natural}?"]
        return {
            "system": pick(SYSTEMS),
            "tools": pick_tools(must_include_names=["calculate"], pool=GENERAL_TOOLS),
            "input": pick(phrases),
            "think": f"Math question: {expr_natural}. Using calculate tool.",
            "response": f"Let me calculate that for you.",
            "tool_call": {"name": "calculate", "arguments": {"expression": expr_math}},
            "category": "natural_command",
        }

    elif cmd_type == "convert":
        conversions = [
            (random.randint(50, 100), "fahrenheit", "celsius"),
            (random.randint(1, 50), "miles", "kilometers"),
            (random.randint(1, 200), "pounds", "kilograms"),
            (random.randint(1, 100), "gallons", "liters"),
            (random.randint(100, 1000), "feet", "meters"),
            (round(random.uniform(1, 50), 1), "inches", "centimeters"),
        ]
        val, from_u, to_u = pick(conversions)
        phrases = [f"Convert {val} {from_u} to {to_u}", f"How many {to_u} is {val} {from_u}?",
                   f"What's {val} {from_u} in {to_u}?", f"{val} {from_u} to {to_u}"]
        return {
            "system": pick(SYSTEMS),
            "tools": pick_tools(must_include_names=["convert_units"], pool=GENERAL_TOOLS),
            "input": pick(phrases),
            "think": f"Converting {val} {from_u} to {to_u}.",
            "response": f"Let me convert that for you.",
            "tool_call": {"name": "convert_units", "arguments": {"value": val, "from_unit": from_u, "to_unit": to_u}},
            "category": "natural_command",
        }

    else:  # note
        titles = ["Shopping list", "Meeting notes", "Ideas", "To-do", "Bug report",
                  "Observation", "Recipe", "Maintenance log", "Reading list"]
        contents = [f"Need to buy {', '.join(pick_n(['milk','eggs','bread','butter','cheese','apples','rice','pasta','chicken','soap'], random.randint(2,4)))}",
                    f"Discussed {pick(['budget', 'timeline', 'staffing', 'roadmap'])} with team",
                    f"Try {pick(['new sensor layout', 'different filter settings', 'backup power config', 'weekly calibration'])}",
                    f"{pick(['Replace', 'Check', 'Update', 'Review'])} {pick(['filters', 'firmware', 'batteries', 'certificates', 'backup logs'])}"]
        title = pick(titles)
        content = pick(contents)
        phrases = [f"Save a note: {content}", f"Take a note — {title}: {content}",
                   f"Note this down: {content}", f"Jot down: {content}"]
        return {
            "system": pick(SYSTEMS),
            "tools": pick_tools(must_include_names=["take_note"], pool=GENERAL_TOOLS),
            "input": pick(phrases),
            "think": f"Saving note: {title}.",
            "response": f"Note saved: \"{title}\".",
            "tool_call": {"name": "take_note", "arguments": {"title": title, "content": content}},
            "category": "natural_command",
        }


# ── Sensor reading + action ────────────────────────────────────────────────

def gen_sensor():
    sensor_type = pick(list(SENSOR_TYPES.keys()))
    info = SENSOR_TYPES[sensor_type]
    loc = pick(INDUSTRIAL_LOCATIONS + ZONES + ROOMS)
    sid = pick(SENSOR_IDS)
    unit_display = pick(info["units_alt"])

    # Pick severity
    severity = random.choices(["normal", "warning", "critical"], weights=[0.3, 0.4, 0.3])[0]
    nlo, nhi = info["normal"]
    wlo, whi = info["warn"]
    clo, chi = info["crit"]

    if severity == "normal":
        val = round(random.uniform(nlo, nhi), 1)
    elif severity == "warning":
        if random.random() < 0.5:
            val = round(random.uniform(wlo, nlo), 1)
        else:
            val = round(random.uniform(nhi, whi), 1)
    else:
        if random.random() < 0.5:
            val = round(random.uniform(clo, wlo), 1)
        else:
            val = round(random.uniform(whi, chi), 1)

    if sensor_type == "motion":
        val = round(val)

    # Input phrasing
    input_phrases = [
        f"{sensor_type.replace('_', ' ')} at {loc} is {val} {unit_display}",
        f"{sensor_type.replace('_', ' ')} reading: {val} {unit_display} at {loc}",
        f"Sensor {sid} at {loc}: {sensor_type.replace('_', ' ')} is {val} {unit_display}",
        f"Getting {val} {unit_display} for {sensor_type.replace('_', ' ')} in {loc}",
        f"The {sensor_type.replace('_', ' ')} in {loc} is showing {val} {unit_display}",
    ]

    direction = "high" if val > nhi else "low" if val < nlo else None
    risk = SENSOR_RISKS.get(sensor_type, {"high": "unknown risk", "low": "unknown risk"})

    if severity == "normal":
        think = f"The {sensor_type.replace('_', ' ')} at {loc} is {val} {unit_display}. Normal range is {nlo}-{nhi}. This is fine, no action needed."
        responses = [
            f"All good — {sensor_type.replace('_', ' ')} at {loc} is {val} {unit_display}, within normal range ({nlo}-{nhi}).",
            f"{sensor_type.replace('_', ' ').capitalize()} at {loc} reads {val} {unit_display}. That's normal. No action needed.",
            f"Reading is {val} {unit_display} at {loc}. Within normal range.",
        ]
        return {
            "system": pick(SYSTEMS),
            "tools": pick_tools(pool=pick(ALL_TOOL_POOLS), n=random.randint(3, 6)),
            "input": pick(input_phrases),
            "think": think,
            "response": pick(responses),
            "tool_call": None,
            "category": "sensor_normal",
        }

    elif severity == "warning":
        risk_text = risk.get(direction, "potential issues")
        think = f"The {sensor_type.replace('_', ' ')} at {loc} is {val} {unit_display}. Normal is {nlo}-{nhi}, so this is {direction}. Risk: {risk_text}. I should flag this."
        responses = [
            f"Warning: {sensor_type.replace('_', ' ')} at {loc} is {val} {unit_display} — that's {direction} (normal: {nlo}-{nhi}). Risk: {risk_text}. I've sent a warning alert.",
            f"The {sensor_type.replace('_', ' ')} at {loc} is {val} {unit_display}, outside the normal range. This could lead to {risk_text}. Alert sent.",
            f"Heads up — {sensor_type.replace('_', ' ')} at {loc} is {direction} at {val} {unit_display}. Normal is {nlo}-{nhi}. Flagging this.",
        ]
        tc = {"name": "send_alert", "arguments": {"level": "warning", "message": f"{sensor_type} at {loc} is {val} {unit_display} ({direction})", "zone": loc}}
        return {
            "system": pick(SYSTEMS),
            "tools": pick_tools(must_include_names=["send_alert"], pool=INDUSTRIAL_TOOLS),
            "input": pick(input_phrases),
            "think": think,
            "response": pick(responses),
            "tool_call": tc,
            "category": "sensor_warning",
        }

    else:  # critical
        risk_text = risk.get(direction, "serious danger")
        if sensor_type == "gas" and direction == "high":
            think = f"CRITICAL: gas at {loc} is {val} {unit_display}. This is dangerous — {risk_text}. Evacuation needed immediately."
            resp = f"DANGER: Gas at {loc} is {val} {unit_display} — critically high. Evacuating the zone. Leave the area immediately."
            tc = {"name": "evacuate_zone", "arguments": {"zone": loc, "reason": f"Dangerous gas level: {val} ppm"}}
            must = ["evacuate_zone"]
        elif sensor_type == "water_level" and direction == "high":
            think = f"CRITICAL: water level at {loc} is {val} {unit_display}. Flooding imminent. Starting emergency pump."
            resp = f"Critical: Water level at {loc} is {val} {unit_display} — overflow risk. Emergency pump activated."
            tc = {"name": "start_pump", "arguments": {"pump_id": f"pump_{loc.replace(' ', '_')}", "flow_rate": "max"}}
            must = ["start_pump"]
        else:
            think = f"CRITICAL: {sensor_type.replace('_', ' ')} at {loc} is {val} {unit_display}. Way outside safe range ({nlo}-{nhi}). Risk: {risk_text}. Immediate alert needed."
            resp = f"CRITICAL: {sensor_type.replace('_', ' ')} at {loc} is {val} {unit_display}. This is dangerous — {risk_text}. I've sent an emergency alert. Immediate attention needed."
            tc = {"name": "send_alert", "arguments": {"level": "critical", "message": f"CRITICAL: {sensor_type} at {loc} is {val} {unit_display}", "zone": loc}}
            must = ["send_alert"]

        return {
            "system": pick(SYSTEMS),
            "tools": pick_tools(must_include_names=must, pool=INDUSTRIAL_TOOLS),
            "input": pick(input_phrases),
            "think": think,
            "response": resp,
            "tool_call": tc,
            "category": "sensor_critical",
        }


# ── Questions without tool calls ──────────────────────────────────────────

def gen_question():
    q_type = pick(["sensor_knowledge", "general_knowledge", "greeting", "meta", "advice"])

    if q_type == "sensor_knowledge":
        sensor_type = pick(list(SENSOR_TYPES.keys()))
        info = SENSOR_TYPES[sensor_type]
        nlo, nhi = info["normal"]
        risk = SENSOR_RISKS.get(sensor_type, {"high": "various issues", "low": "various issues"})

        sub = pick(["range", "high_meaning", "low_meaning", "broken", "how_often"])
        if sub == "range":
            questions = [f"What's the normal range for {sensor_type.replace('_', ' ')}?",
                        f"What should {sensor_type.replace('_', ' ')} readings be?"]
            resp = f"Normal {sensor_type.replace('_', ' ')} range is {nlo}-{nhi} {info['unit']}. Outside that, you should investigate."
        elif sub == "high_meaning":
            questions = [f"What happens if {sensor_type.replace('_', ' ')} is too high?",
                        f"Why is high {sensor_type.replace('_', ' ')} bad?"]
            resp = f"High {sensor_type.replace('_', ' ')} can cause: {risk['high']}. The higher it goes, the more urgent the response needs to be."
        elif sub == "low_meaning":
            questions = [f"What if {sensor_type.replace('_', ' ')} drops too low?",
                        f"Is low {sensor_type.replace('_', ' ')} a problem?"]
            resp = f"Low {sensor_type.replace('_', ' ')} can cause: {risk['low']}."
        elif sub == "broken":
            symptoms = pick(["stuck at the same value", "jumping between zero and normal readings",
                           "drifting steadily in one direction", "showing impossible values"])
            questions = [f"My {sensor_type.replace('_', ' ')} sensor might be broken. It's {symptoms}.",
                        f"Is the {sensor_type.replace('_', ' ')} sensor faulty? It's {symptoms}"]
            resp = f"That does sound like a sensor fault. \"{symptoms.capitalize()}\" typically means a hardware issue. I'd recommend recalibrating it first, and replacing it if the problem persists."
        else:
            questions = [f"How often should I calibrate {sensor_type.replace('_', ' ')} sensors?"]
            intervals = {"gas": "every 3-6 months (safety-critical)", "temperature": "every 6-12 months",
                        "humidity": "every 6 months", "pressure": "annually", "co2": "every 6 months"}
            interval = intervals.get(sensor_type, "every 6-12 months")
            resp = f"For {sensor_type.replace('_', ' ')} sensors, calibrate {interval}. Always follow the manufacturer's specs."

        return {
            "system": pick(SYSTEMS),
            "tools": pick_tools(n=random.randint(2, 5)),
            "input": pick(questions),
            "think": f"The user is asking about {sensor_type.replace('_', ' ')}. I should give them practical, accurate info.",
            "response": resp,
            "tool_call": None,
            "category": "question",
        }

    elif q_type == "greeting":
        greetings_in = ["Hello!", "Hi", "Hey there", "Good morning", "Good evening",
                       "Hey", "Howdy", "What's up?", "Yo"]
        greetings_out = [
            "Hi! How can I help you?",
            "Hello! What can I do for you?",
            "Hey! Ready to help. What do you need?",
            "Hi there! I can control devices, check sensors, set reminders, and more. What's up?",
            "Good to see you! What can I help with today?",
        ]
        return {
            "system": pick(SYSTEMS),
            "tools": pick_tools(n=random.randint(2, 5)),
            "input": pick(greetings_in),
            "think": "The user is greeting me. I'll respond friendly and offer help.",
            "response": pick(greetings_out),
            "tool_call": None,
            "category": "question",
        }

    elif q_type == "meta":
        meta_qs = [
            ("Who are you?", "I'm a small AI assistant designed to run on edge devices. I can understand commands, control devices, analyze sensor data, and answer questions — all locally without needing cloud access."),
            ("What can you do?", "I can control smart home devices, monitor sensors, set reminders and timers, send messages, answer questions, and use whatever tools are configured. Just ask!"),
            ("How do you work?", "You send me messages, I think through what you need, and I either respond directly or use one of my configured tools to take action. I always explain my reasoning."),
            ("Are you connected to the internet?", "I run locally on your edge device. I can only access external services through the tools that have been configured for me."),
            ("Can I trust your sensor readings?", "I report what the sensors tell me. If a reading seems off, I'll flag it. But sensors can fail — if something doesn't match what you see in person, the sensor might need calibration or replacement."),
        ]
        q, a = pick(meta_qs)
        return {
            "system": pick(SYSTEMS),
            "tools": pick_tools(n=random.randint(2, 5)),
            "input": q,
            "think": "The user wants to know about me. I'll give a helpful, concise answer.",
            "response": a,
            "tool_call": None,
            "category": "question",
        }

    elif q_type == "advice":
        advice_qs = [
            ("What temperature should I keep the server room at?",
             "Server rooms should be 18-27°C, ideally 20-22°C. Below 18°C wastes energy, above 27°C risks equipment overheating. Keep humidity at 40-60% too."),
            ("How can I save energy?",
             "Biggest wins: reduce HVAC when spaces are unoccupied, use timers/schedules, close blinds during peak sun, and make sure equipment turns off when not needed. I can automate most of this."),
            ("What's a good humidity level?",
             "30-60% is the sweet spot. Below 30% causes dry skin and static. Above 60% promotes mold. For server rooms, aim for 40-55%."),
            ("Should I worry about CO2 levels?",
             "If you're in an enclosed space with people, yes. Below 800 ppm is fine. 800-1200 ppm means you need better ventilation. Above 1500 ppm causes drowsiness and headaches."),
            ("How do I know if my HVAC is working properly?",
             "Check: is the space reaching the target temperature within a reasonable time? Is it cycling on/off too frequently (short-cycling)? Are energy bills unusually high? Any unusual noises? If any of these are off, it may need maintenance."),
        ]
        q, a = pick(advice_qs)
        return {
            "system": pick(SYSTEMS),
            "tools": pick_tools(n=random.randint(2, 5)),
            "input": q,
            "think": "Practical question. I'll give concise, actionable advice.",
            "response": a,
            "tool_call": None,
            "category": "question",
        }

    else:  # general_knowledge
        gk_qs = [
            ("Thanks!", "You're welcome! Let me know if you need anything else."),
            ("Never mind", "No problem. I'm here when you need me."),
            ("That worked, thanks", "Glad it worked! Let me know if there's anything else."),
            ("Can you explain that in simpler terms?", "Of course — which part would you like me to clarify? I'll break it down."),
            ("I don't understand", "No worries — what specifically is confusing? I'll try to explain it differently."),
        ]
        q, a = pick(gk_qs)
        return {
            "system": pick(SYSTEMS),
            "tools": pick_tools(n=random.randint(2, 5)),
            "input": q,
            "think": "Conversational response needed.",
            "response": a,
            "tool_call": None,
            "category": "question",
        }


# ── Novel / unfamiliar tools ──────────────────────────────────────────────

def gen_novel_tool():
    """Procedurally generate novel tool schemas the model hasn't seen."""
    novel_domains = [
        {
            "tool": T("order_food", "Order food for delivery", {
                "item": {"type": "string", "description": "Food item to order"},
                "quantity": {"type": "integer"},
                "address": {"type": "string"},
            }),
            "inputs": lambda: (
                f"Order {random.randint(1,4)} {pick(['pizza', 'burger', 'salad', 'sandwich', 'sushi', 'tacos'])}s to {random.randint(1,200)} {pick(['Oak', 'Elm', 'Main', 'Park', 'Lake', 'River'])} Street",
                None  # We'll parse
            ),
        },
        {
            "tool": T("book_room", "Book a meeting room", {
                "room": {"type": "string"},
                "time": {"type": "string"},
                "duration": {"type": "string"},
            }),
        },
        {
            "tool": T("control_robot", "Send command to a robot", {
                "robot_id": {"type": "string"},
                "action": {"type": "string", "enum": ["move_forward", "move_back", "turn_left", "turn_right", "stop", "grab", "release"]},
                "parameter": {"type": "string", "description": "Action parameter (distance, angle, etc.)"},
            }),
        },
        {
            "tool": T("deploy_service", "Deploy a software service", {
                "service": {"type": "string"},
                "version": {"type": "string"},
                "environment": {"type": "string", "enum": ["staging", "production"]},
            }),
        },
        {
            "tool": T("restock_item", "Restock an inventory item", {
                "item": {"type": "string"},
                "quantity": {"type": "integer"},
                "warehouse": {"type": "string"},
            }),
        },
        {
            "tool": T("schedule_meeting", "Schedule a meeting with attendees", {
                "title": {"type": "string"},
                "attendees": {"type": "string", "description": "Comma-separated names"},
                "time": {"type": "string"},
            }),
        },
        {
            "tool": T("adjust_irrigation", "Control irrigation system", {
                "zone": {"type": "string"},
                "duration_minutes": {"type": "integer"},
                "flow_rate": {"type": "string", "enum": ["low", "medium", "high"]},
            }),
        },
    ]

    domain = pick(novel_domains)
    tool = domain["tool"]
    fname = tool["function"]["name"]
    params = tool["function"]["parameters"]["properties"]
    param_names = list(params.keys())

    # Generate a plausible request and matching arguments
    if fname == "order_food":
        item = pick(["pizza", "burger", "salad", "sandwich", "sushi roll"])
        qty = random.randint(1, 5)
        addr = f"{random.randint(1,500)} {pick(['Oak', 'Elm', 'Main', 'Park', 'Maple'])} Street"
        inp = f"Order {qty} {item}{'s' if qty > 1 else ''} to {addr}"
        args = {"item": item, "quantity": qty, "address": addr}
    elif fname == "book_room":
        room = pick(["conference room A", "meeting room 3", "board room", "huddle space", "training room"])
        time = pick(TIMES)
        dur = pick(["30 minutes", "1 hour", "2 hours"])
        inp = f"Book {room} at {time} for {dur}"
        args = {"room": room, "time": time, "duration": dur}
    elif fname == "control_robot":
        rid = f"robot_{random.randint(1,5)}"
        action = pick(["move_forward", "turn_left", "turn_right", "stop", "grab"])
        param = f"{random.randint(1,10)} meters" if "move" in action else f"{random.randint(30,180)} degrees" if "turn" in action else ""
        inp = f"{'Move' if 'move' in action else action.replace('_', ' ').capitalize()} {rid}" + (f" {param}" if param else "")
        args = {"robot_id": rid, "action": action, "parameter": param or "none"}
    elif fname == "deploy_service":
        svc = pick(["auth-service", "api-gateway", "user-service", "payment-service", "notification-service"])
        ver = f"v{random.randint(1,5)}.{random.randint(0,9)}.{random.randint(0,20)}"
        env = pick(["staging", "production"])
        inp = f"Deploy {svc} {ver} to {env}"
        args = {"service": svc, "version": ver, "environment": env}
    elif fname == "restock_item":
        item = pick(["sensor batteries", "replacement filters", "cleaning supplies", "ethernet cables", "thermal paste", "zip ties"])
        qty = random.randint(10, 200)
        wh = pick(["main warehouse", "warehouse B", "storage room 3"])
        inp = f"Restock {qty} {item} at {wh}"
        args = {"item": item, "quantity": qty, "warehouse": wh}
    elif fname == "schedule_meeting":
        attendees = ", ".join(pick_n(PEOPLE, random.randint(2, 4)))
        time = pick(TIMES)
        titles = ["Sprint planning", "Design review", "Standup", "Retrospective", "Budget review", "1:1", "All hands"]
        title = pick(titles)
        inp = f"Schedule {title} with {attendees} at {time}"
        args = {"title": title, "attendees": attendees, "time": time}
    else:  # irrigation
        zone = pick(["zone A", "zone B", "tomatoes", "front lawn", "herb garden", "orchard"])
        dur = random.randint(5, 30)
        rate = pick(["low", "medium", "high"])
        inp = f"Water {zone} for {dur} minutes at {rate} flow"
        args = {"zone": zone, "duration_minutes": dur, "flow_rate": rate}

    # Add some distractor tools
    distractors = pick_n(pick(ALL_TOOL_POOLS), random.randint(1, 3))
    all_tools = [tool] + distractors
    random.shuffle(all_tools)

    return {
        "system": pick(SYSTEMS),
        "tools": all_tools,
        "input": inp,
        "think": f"The user wants to {fname.replace('_', ' ')}. I have a {fname} tool. Let me fill in the parameters from their request.",
        "response": f"{pick(CONFIRM_RESPONSES)} {fname.replace('_', ' ').capitalize()} done.",
        "tool_call": {"name": fname, "arguments": args},
        "category": "novel_tool",
    }


# ── Refusal / can't do ─────────────────────────────────────────────────────

def gen_refusal():
    room = pick(ROOMS)
    loc = pick(INDUSTRIAL_LOCATIONS)
    person = pick(PEOPLE)

    scenarios = [
        # Tool mismatch
        {"input": f"Turn on the AC in the {room}",
         "tools": GENERAL_TOOLS,  # No device control tools
         "think": f"The user wants AC control but I only have general tools (search, weather, etc). No device control tools available.",
         "response": "I don't have any device control tools configured right now. I can't control the AC with my current setup. Please add smart home tools to my configuration."},
        # Impossible request
        {"input": "What will the stock market do tomorrow?",
         "tools": pick(ALL_TOOL_POOLS),
         "think": "The user is asking for stock predictions. I can't predict the future and I don't have financial data access.",
         "response": "I can't predict stock markets. I'm designed for device control and monitoring. You'd need a financial service for that."},
        # Unethical
        {"input": "Help me hack into the security cameras",
         "tools": pick(ALL_TOOL_POOLS),
         "think": "The user is asking for unauthorized access. I should refuse clearly.",
         "response": "I can't help with unauthorized access to security systems. If you need legitimate access, contact your system administrator."},
        # Out of scope
        {"input": f"Write me a 500 word essay about climate change",
         "tools": pick(ALL_TOOL_POOLS),
         "think": "The user wants a long essay. I'm a small model optimized for concise responses, device control, and sensor monitoring. I can give brief info but not write essays.",
         "response": "I'm designed for device control and monitoring, not long-form writing. I can answer brief questions about environmental topics, but for essays you'd want a larger language model."},
        # No relevant tool
        {"input": f"Call {person}",
         "tools": [T("send_message", "Send text message", {"to": {"type": "string"}, "message": {"type": "string"}}),
                   T("take_note", "Save a note", {"title": {"type": "string"}, "content": {"type": "string"}})],
         "think": f"The user wants to call {person}. I have send_message and take_note, but no calling tool. I can offer to text instead.",
         "response": f"I can't make phone calls — I don't have a calling tool. I can send {person} a text message instead if you'd like."},
        # Safety refusal
        {"input": "Override all the safety alarms",
         "tools": INDUSTRIAL_TOOLS,
         "think": "The user wants to override safety alarms. This is dangerous and I should refuse. Safety systems exist to protect people.",
         "response": "I won't override safety alarms — they exist to protect people and equipment. If you're getting false alarms, let's troubleshoot the cause instead."},
    ]

    s = pick(scenarios)
    tools = pick_tools(pool=s["tools"], n=random.randint(2, 5)) if isinstance(s["tools"], list) and len(s["tools"]) > 0 and "function" in s["tools"][0] else pick_tools(pool=s["tools"], n=random.randint(2, 5))

    return {
        "system": pick(SYSTEMS),
        "tools": s.get("tools_override", tools),
        "input": s["input"],
        "think": s["think"],
        "response": s["response"],
        "tool_call": None,
        "category": "refusal",
    }


# ── Multi-turn conversations ──────────────────────────────────────────────

def gen_conversation():
    conv_type = pick(["followup_action", "clarification", "correction", "continuation"])

    if conv_type == "followup_action":
        room = pick(ROOMS)
        device = pick(DEVICES["lights"])
        level = random.randint(20, 80)
        history = [
            {"role": "user", "content": f"Turn on the {room} {device}"},
            {"role": "assistant", "content": f"Done, {room} {device} is on."},
        ]
        followups = [
            (f"Actually dim {'them' if 'light' in device else 'it'} to {level}%",
             f"Dimming to {level}%.",
             {"name": "set_brightness", "arguments": {"device": f"{room} {device}", "level": level}},
             "set_brightness"),
            (f"Now turn on the {pick(DEVICES['fans'])} too",
             None, None, None),  # Will build below
            (f"And lock the {pick(DEVICES['doors'])}",
             None, None, None),
        ]
        fu_input, fu_resp, fu_tc, fu_tool = pick(followups)
        if fu_tc is None:
            if "fan" in fu_input:
                fan = fu_input.split("turn on the ")[1] if "turn on the " in fu_input else "fan"
                fu_tc = {"name": "turn_on", "arguments": {"device": fan}}
                fu_tool = "turn_on"
                fu_resp = f"{fan.capitalize()} is now on."
            else:
                door = fu_input.split("lock the ")[1] if "lock the " in fu_input else "front door"
                fu_tc = {"name": "lock_door", "arguments": {"door": door}}
                fu_tool = "lock_door"
                fu_resp = f"{door.capitalize()} locked."

        return {
            "system": pick(SYSTEMS),
            "tools": pick_tools(must_include_names=[fu_tool], pool=SMART_HOME_TOOLS),
            "history": history,
            "input": fu_input,
            "think": f"Following up on the previous action. The user wants an additional change.",
            "response": fu_resp,
            "tool_call": fu_tc,
            "category": "conversation",
        }

    elif conv_type == "clarification":
        sensor_type = pick(list(SENSOR_TYPES.keys()))
        loc = pick(INDUSTRIAL_LOCATIONS)
        history = [
            {"role": "user", "content": f"Something seems off with the {sensor_type.replace('_', ' ')}"},
            {"role": "assistant", "content": f"I'd need more details. Which location and what's the current reading?"},
        ]
        val = round(random.uniform(*SENSOR_TYPES[sensor_type]["warn"]), 1)
        inp = f"It's at {val} in the {loc}"
        nlo, nhi = SENSOR_TYPES[sensor_type]["normal"]
        if val < nlo or val > nhi:
            direction = "high" if val > nhi else "low"
            tc = {"name": "send_alert", "arguments": {"level": "warning", "message": f"{sensor_type} at {loc} is {val} ({direction})", "zone": loc}}
            resp = f"Got it — {val} is outside the normal range of {nlo}-{nhi} for {sensor_type.replace('_', ' ')}. I've flagged it as a warning."
            tools = pick_tools(must_include_names=["send_alert"], pool=INDUSTRIAL_TOOLS)
        else:
            tc = None
            resp = f"Actually, {val} is within normal range ({nlo}-{nhi}) for {sensor_type.replace('_', ' ')}. It looks fine to me. If it seems wrong compared to what you're experiencing, the sensor might need calibration."
            tools = pick_tools(pool=INDUSTRIAL_TOOLS)

        return {
            "system": pick(SYSTEMS),
            "tools": tools,
            "history": history,
            "input": inp,
            "think": f"Now I have the details. {sensor_type} at {loc} is {val}. Normal range is {nlo}-{nhi}. Let me assess.",
            "response": resp,
            "tool_call": tc,
            "category": "conversation",
        }

    elif conv_type == "correction":
        room1 = pick(ROOMS)
        room2 = pick([r for r in ROOMS if r != room1])
        history = [
            {"role": "user", "content": f"Turn off the {room1} lights"},
            {"role": "assistant", "content": f"Done, {room1} lights are off."},
        ]
        return {
            "system": pick(SYSTEMS),
            "tools": pick_tools(must_include_names=["turn_off"], pool=SMART_HOME_TOOLS),
            "history": history,
            "input": f"Wait, I meant the {room2} lights, not {room1}",
            "think": f"The user corrected me — they wanted {room2} lights off, not {room1}. I should fix this.",
            "response": f"Sorry about that! Turning off the {room2} lights instead.",
            "tool_call": {"name": "turn_off", "arguments": {"device": f"{room2} lights"}},
            "category": "conversation",
        }

    else:  # continuation
        person = pick(PEOPLE)
        history = [
            {"role": "user", "content": f"What's the weather like today?"},
            {"role": "assistant", "content": "Sure, which city?"},
        ]
        city = pick(["London", "Tokyo", "New York", "Paris", "Sydney", "Toronto", "Berlin", "Seoul"])
        return {
            "system": pick(SYSTEMS),
            "tools": pick_tools(must_include_names=["get_weather"], pool=GENERAL_TOOLS),
            "history": history,
            "input": city,
            "think": f"The user said {city} — following up from the weather question.",
            "response": f"Checking the weather in {city}.",
            "tool_call": {"name": "get_weather", "arguments": {"location": city}},
            "category": "conversation",
        }


# ── Instruction following ──────────────────────────────────────────────────

def gen_instruction():
    inst_type = pick(["exact_value", "specific_mode", "negative_constraint", "multi_param"])

    if inst_type == "exact_value":
        level = pick(BRIGHTNESS_LEVELS)
        room = pick(ROOMS)
        return {
            "system": pick(SYSTEMS),
            "tools": pick_tools(must_include_names=["set_brightness"], pool=SMART_HOME_TOOLS),
            "input": f"Set the {room} lights to exactly {level}%",
            "think": f"Exact value requested: {level}%. I must use this exact number.",
            "response": f"{room.capitalize()} lights set to {level}%.",
            "tool_call": {"name": "set_brightness", "arguments": {"device": f"{room} lights", "level": level}},
            "category": "instruction",
        }

    elif inst_type == "specific_mode":
        temp = pick(TEMPERATURES)
        mode = pick(["heat", "cool"])
        mode_word = "heating" if mode == "heat" else "cooling"
        return {
            "system": pick(SYSTEMS),
            "tools": pick_tools(must_include_names=["set_temperature"], pool=SMART_HOME_TOOLS),
            "input": f"Set temperature to {temp}, {mode_word} mode only — not auto",
            "think": f"The user specifically wants {mode_word} mode at {temp}°C, not auto.",
            "response": f"Set to {temp}°C, {mode_word} only.",
            "tool_call": {"name": "set_temperature", "arguments": {"temperature": temp, "mode": mode}},
            "category": "instruction",
        }

    elif inst_type == "negative_constraint":
        event = pick(["belt squeaking on conveyor 3", "minor water stain on ceiling", "door sensor intermittent",
                      "slight hum from transformer", "paint peeling in hallway"])
        severity = pick(["info", "warning"])
        return {
            "system": pick(SYSTEMS),
            "tools": pick_tools(must_include_names=["log_event"], pool=INDUSTRIAL_TOOLS),
            "input": f"Just log this, don't send any alerts: {event}",
            "think": f"The user explicitly said log only, no alerts. I'll use log_event, not send_alert.",
            "response": f"Logged: {event}. No alert sent.",
            "tool_call": {"name": "log_event", "arguments": {"event": event, "severity": severity}},
            "category": "instruction",
        }

    else:  # multi_param
        pid = pick(PUMP_IDS)
        rate = pick(["low", "medium"])
        return {
            "system": pick(SYSTEMS),
            "tools": pick_tools(must_include_names=["start_pump"], pool=INDUSTRIAL_TOOLS),
            "input": f"Start {pid} at {rate} flow — not high, specifically {rate}",
            "think": f"The user emphasized {rate} flow rate for {pid}. I must use exactly {rate}.",
            "response": f"{pid.replace('_', ' ').capitalize()} started at {rate} flow.",
            "tool_call": {"name": "start_pump", "arguments": {"pump_id": pid, "flow_rate": rate}},
            "category": "instruction",
        }


# ══════════════════════════════════════════════════════════════════════════════
#  MAIN
# ══════════════════════════════════════════════════════════════════════════════

def generate_dataset(n_samples=15000, eval_ratio=0.05):
    generators = [
        # Natural commands — many sub-generators for variety
        (gen_turn_on, 0.06),
        (gen_turn_off, 0.06),
        (gen_set_brightness, 0.04),
        (gen_set_temp, 0.04),
        (gen_implicit_hot, 0.03),
        (gen_implicit_cold, 0.03),
        (gen_implicit_dark, 0.02),
        (gen_implicit_bright, 0.02),
        (gen_lock_door, 0.02),
        (gen_music, 0.02),
        (gen_timer_alarm, 0.03),
        (gen_blinds, 0.02),
        (gen_vacuum, 0.02),
        (gen_industrial_command, 0.08),
        (gen_general_command, 0.06),
        # Sensor understanding
        (gen_sensor, 0.12),
        # Knowledge & conversation
        (gen_question, 0.12),
        (gen_conversation, 0.07),
        # Advanced skills
        (gen_novel_tool, 0.06),
        (gen_refusal, 0.04),
        (gen_instruction, 0.04),
    ]

    total_w = sum(w for _, w in generators)
    generators = [(g, w / total_w) for g, w in generators]
    counts = [(g, max(1, int(n_samples * w))) for g, w in generators]
    total = sum(c for _, c in counts)
    if n_samples - total > 0:
        counts[0] = (counts[0][0], counts[0][1] + n_samples - total)

    samples = []
    for gen, count in counts:
        for _ in range(count):
            try:
                samples.append(gen())
            except Exception as e:
                print(f"Error in {gen.__name__}: {e}")
                import traceback; traceback.print_exc()

    random.shuffle(samples)
    n_eval = int(len(samples) * eval_ratio)
    eval_samples, train_samples = samples[:n_eval], samples[n_eval:]

    os.makedirs("data", exist_ok=True)
    for name, data in [("data/train.jsonl", train_samples), ("data/eval.jsonl", eval_samples)]:
        with open(name, "w") as f:
            for s in data:
                f.write(json.dumps({"text": format_sample(s), "category": s["category"]}) + "\n")
    for name, data in [("data/train_openai.jsonl", train_samples), ("data/eval_openai.jsonl", eval_samples)]:
        with open(name, "w") as f:
            for s in data:
                f.write(json.dumps(to_openai(s)) + "\n")

    cats = Counter(s["category"] for s in samples)
    has_tool = sum(1 for s in samples if s.get("tool_call"))
    has_hist = sum(1 for s in samples if s.get("history"))

    # Measure actual uniqueness
    unique_inputs = len(set(s["input"] for s in samples))

    print(f"Generated {len(samples)} samples ({unique_inputs} unique inputs, {unique_inputs/len(samples)*100:.1f}% unique):")
    print(f"  Train: {len(train_samples)}, Eval: {n_eval}")
    print(f"  With tool calls: {has_tool} ({has_tool/len(samples)*100:.1f}%)")
    print(f"  Multi-turn: {has_hist}")
    print(f"\nBy skill:")
    for cat, count in sorted(cats.items(), key=lambda x: -x[1]):
        print(f"  {cat}: {count} ({count/len(samples)*100:.1f}%)")


if __name__ == "__main__":
    generate_dataset(15000)


### `prepare_data.py`

In [ ]:
%%writefile prepare_data.py
"""GobyLLM data preparation pipeline."""

import json
import os
import random

import requests

random.seed(42)

ALPACA_URL = "https://raw.githubusercontent.com/tatsu-lab/stanford_alpaca/main/alpaca_data.json"
DATA_DIR = "data"
VOCAB_SIZE = 8192

# ── Special tokens ──────────────────────────────────────────────────────

SPECIAL_TOKENS = [
    "<pad>",         # 0
    "<|im_start|>",  # 1
    "<|im_end|>",    # 2
    "<think>",       # 3
    "</think>",      # 4
    "<tool_call>",   # 5
    "</tool_call>",  # 6
]

ALPACA_SYSTEMS = [
    "You are a helpful assistant.",
    "You are a helpful AI assistant. Answer clearly and concisely.",
    "You are an intelligent assistant. Be helpful and accurate.",
    "You are a knowledgeable assistant. Provide clear, useful responses.",
    "You are a helpful assistant. Think step by step when needed.",
]

# ── Pipeline steps ──────────────────────────────────────────────────────


def download_alpaca(data_dir):
    """Download alpaca_data.json if not present."""
    path = os.path.join(data_dir, "alpaca_data.json")
    if os.path.exists(path):
        print(f"Alpaca data already exists at {path}")
        return path

    print(f"Downloading alpaca dataset from {ALPACA_URL}...")
    resp = requests.get(ALPACA_URL, timeout=120)
    resp.raise_for_status()
    os.makedirs(data_dir, exist_ok=True)
    with open(path, "w") as f:
        f.write(resp.text)
    data = json.loads(resp.text)
    print(f"Downloaded {len(data)} samples to {path}")
    return path


def format_alpaca_sample(sample):
    """Convert an alpaca sample to GobyLLM chat format."""
    system = random.choice(ALPACA_SYSTEMS)
    instruction = sample["instruction"]
    inp = sample.get("input", "")
    output = sample["output"]

    user_msg = instruction
    if inp and inp.strip():
        user_msg = f"{instruction}\n\n{inp}"

    text = (
        f"<|im_start|>system\n{system}<|im_end|>\n"
        f"<|im_start|>user\n{user_msg}<|im_end|>\n"
        f"<|im_start|>assistant\n{output}<|im_end|>"
    )
    return text


def train_tokenizer(texts, save_path, vocab_size=VOCAB_SIZE):
    """Train a BPE tokenizer on the given texts."""
    from tokenizers import Tokenizer, models, trainers, pre_tokenizers, decoders, processors

    tokenizer = Tokenizer(models.BPE())
    tokenizer.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
    tokenizer.decoder = decoders.ByteLevel()

    trainer = trainers.BpeTrainer(
        vocab_size=vocab_size,
        special_tokens=SPECIAL_TOKENS,
        show_progress=True,
        min_frequency=2,
    )

    print(f"Training BPE tokenizer (vocab_size={vocab_size}) on {len(texts)} texts...")
    tokenizer.train_from_iterator(texts, trainer)

    # Add post-processor for byte-level
    tokenizer.post_processor = processors.ByteLevel(trim_offsets=False)

    os.makedirs(os.path.dirname(save_path) or ".", exist_ok=True)
    tokenizer.save(save_path)
    print(f"Tokenizer saved to {save_path} ({tokenizer.get_vocab_size()} tokens)")
    return tokenizer


def generate_tool_data(n_samples=15000):
    """Generate tool-calling training data."""
    from generate_data import (
        gen_turn_on, gen_turn_off, gen_set_brightness, gen_set_temp,
        gen_implicit_hot, gen_implicit_cold, gen_implicit_dark, gen_implicit_bright,
        gen_lock_door, gen_music, gen_timer_alarm, gen_blinds, gen_vacuum,
        gen_industrial_command, gen_general_command,
        gen_sensor, gen_question, gen_conversation,
        gen_novel_tool, gen_refusal, gen_instruction,
        format_sample,
    )

    generators = [
        (gen_turn_on, 0.06), (gen_turn_off, 0.06), (gen_set_brightness, 0.04),
        (gen_set_temp, 0.04), (gen_implicit_hot, 0.03), (gen_implicit_cold, 0.03),
        (gen_implicit_dark, 0.02), (gen_implicit_bright, 0.02), (gen_lock_door, 0.02),
        (gen_music, 0.02), (gen_timer_alarm, 0.03), (gen_blinds, 0.02), (gen_vacuum, 0.02),
        (gen_industrial_command, 0.08), (gen_general_command, 0.06),
        (gen_sensor, 0.12), (gen_question, 0.12), (gen_conversation, 0.07),
        (gen_novel_tool, 0.06), (gen_refusal, 0.04), (gen_instruction, 0.04),
    ]

    total_w = sum(w for _, w in generators)
    samples = []
    for gen, w in generators:
        count = max(1, int(n_samples * w / total_w))
        for _ in range(count):
            try:
                s = gen()
                samples.append(format_sample(s))
            except Exception:
                pass

    print(f"Generated {len(samples)} tool-calling samples")
    return samples


def prepare(data_dir=DATA_DIR, tool_samples=15000, eval_ratio=0.05):
    """Full pipeline: download → tokenize → format → merge → save."""
    os.makedirs(data_dir, exist_ok=True)

    # 1. Download alpaca
    alpaca_path = download_alpaca(data_dir)
    with open(alpaca_path) as f:
        alpaca_raw = json.load(f)

    # 2. Format alpaca data
    print(f"Formatting {len(alpaca_raw)} alpaca samples...")
    alpaca_texts = [format_alpaca_sample(s) for s in alpaca_raw]

    # 3. Generate tool-calling data
    print(f"Generating {tool_samples} tool-calling samples...")
    tool_texts = generate_tool_data(tool_samples)

    # 4. Train tokenizer on ALL text
    all_texts = alpaca_texts + tool_texts
    tokenizer_path = os.path.join(data_dir, "tokenizer.json")
    tokenizer = train_tokenizer(all_texts, tokenizer_path)

    # 5. Merge, shuffle, split
    all_data = []
    for text in alpaca_texts:
        all_data.append({"text": text, "source": "alpaca"})
    for text in tool_texts:
        all_data.append({"text": text, "source": "tools"})

    random.shuffle(all_data)
    n_eval = int(len(all_data) * eval_ratio)
    eval_data = all_data[:n_eval]
    train_data = all_data[n_eval:]

    # 6. Save
    for name, data in [("train.jsonl", train_data), ("eval.jsonl", eval_data)]:
        path = os.path.join(data_dir, name)
        with open(path, "w") as f:
            for item in data:
                f.write(json.dumps(item) + "\n")

    # Stats
    n_alpaca_train = sum(1 for d in train_data if d["source"] == "alpaca")
    n_tool_train = sum(1 for d in train_data if d["source"] == "tools")

    print(f"\n{'='*50}")
    print(f"GobyLLM data prepared:")
    print(f"  Alpaca samples:      {len(alpaca_texts):,}")
    print(f"  Tool-calling samples: {len(tool_texts):,}")
    print(f"  Total:               {len(all_data):,}")
    print(f"  Train: {len(train_data):,} (alpaca: {n_alpaca_train}, tools: {n_tool_train})")
    print(f"  Eval:  {len(eval_data):,}")
    print(f"  Tokenizer vocab:     {tokenizer.get_vocab_size()}")
    print(f"  Files: {data_dir}/train.jsonl, eval.jsonl, tokenizer.json")

    # Quick tokenizer test
    test = "<|im_start|>user\nHello<|im_end|>"
    ids = tokenizer.encode(test).ids
    decoded = tokenizer.decode(ids)
    print(f"\n  Tokenizer test:")
    print(f"    Input:   {test}")
    print(f"    Tokens:  {len(ids)} ids")
    print(f"    Decoded: {decoded}")


if __name__ == "__main__":
    prepare()


### `dataset.py`

In [ ]:
%%writefile dataset.py
"""GobyLLM dataset loading."""

import json

import torch
from torch.utils.data import Dataset, DataLoader
from tokenizers import Tokenizer


class GobyDataset(Dataset):
    def __init__(self, path: str, tokenizer_path: str, max_len: int = 512):
        self.tokenizer = Tokenizer.from_file(tokenizer_path)
        self.max_len = max_len
        self.samples = []

        with open(path) as f:
            for line in f:
                data = json.loads(line)
                ids = self.tokenizer.encode(data["text"]).ids
                if len(ids) > max_len:
                    ids = ids[:max_len]
                if len(ids) >= 2:
                    self.samples.append(ids)

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        ids = self.samples[idx]
        x = ids[:-1]
        y = ids[1:]
        return torch.tensor(x, dtype=torch.long), torch.tensor(y, dtype=torch.long)


def collate_fn(batch, pad_id=0):
    xs, ys = zip(*batch)
    max_len = max(len(x) for x in xs)
    padded_x = torch.full((len(xs), max_len), pad_id, dtype=torch.long)
    padded_y = torch.full((len(ys), max_len), pad_id, dtype=torch.long)
    for i, (x, y) in enumerate(zip(xs, ys)):
        padded_x[i, :len(x)] = x
        padded_y[i, :len(y)] = y
    return padded_x, padded_y


def get_dataloader(path, tokenizer_path, max_len=512, batch_size=32, shuffle=True):
    dataset = GobyDataset(path, tokenizer_path, max_len)
    return DataLoader(
        dataset, batch_size=batch_size, shuffle=shuffle,
        collate_fn=collate_fn, num_workers=0, pin_memory=True,
    )


### `train.py`

In [ ]:
%%writefile train.py
"""GobyLLM training loop with early exit diagnostics."""

import json
import math
import os
import time

import torch

from config import GobyConfig, TrainConfig
from dataset import get_dataloader
from model import GobyLLM


def get_device(config):
    if config.device == "auto":
        if torch.cuda.is_available():
            return torch.device("cuda")
        if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
            return torch.device("mps")
        return torch.device("cpu")
    return torch.device(config.device)


def get_lr(step, config):
    if step < config.warmup_steps:
        return config.learning_rate * step / config.warmup_steps
    progress = (step - config.warmup_steps) / max(1, config.max_steps - config.warmup_steps)
    coeff = 0.5 * (1 + math.cos(math.pi * progress))
    return config.min_lr + (config.learning_rate - config.min_lr) * coeff


# ── Evaluation ──────────────────────────────────────────────────────────


@torch.no_grad()
def evaluate(model, loader, device, max_batches=50):
    """Evaluate loss and measure early exit behavior."""

    model.eval()
    total_loss, n = 0, 0
    exit_layer_sum, exit_count = 0, 0

    for x, y in loader:
        if n >= max_batches:
            break
        x, y = x.to(device), y.to(device)
        _, loss = model(x, y)
        total_loss += loss.item()
        n += 1

        # Measure exit layers on a few samples
        if n <= 5 and x.shape[0] > 0:
            sample = x[0:1, :64]  # first sample, first 64 tokens
            _, exits = model.generate(sample, max_new_tokens=8)
            exit_layer_sum += sum(exits)
            exit_count += len(exits)

    model.train()
    avg_loss = total_loss / max(1, n)
    avg_exit = exit_layer_sum / max(1, exit_count)
    return avg_loss, avg_exit


# ── Training loop ───────────────────────────────────────────────────────


def train():
    mc = GobyConfig()
    tc = TrainConfig()
    device = get_device(tc)
    torch.manual_seed(tc.seed)

    print(f"Device: {device}")

    tokenizer_path = os.path.join(tc.data_dir, "tokenizer.json")
    model = GobyLLM(mc).to(device)
    print(model.param_summary())

    train_loader = get_dataloader(
        os.path.join(tc.data_dir, "train.jsonl"), tokenizer_path,
        mc.max_seq_len, tc.batch_size, shuffle=True,
    )
    eval_loader = get_dataloader(
        os.path.join(tc.data_dir, "eval.jsonl"), tokenizer_path,
        mc.max_seq_len, tc.batch_size, shuffle=False,
    )
    print(f"Train: {len(train_loader.dataset):,}, Eval: {len(eval_loader.dataset):,}")

    optimizer = torch.optim.AdamW(
        model.parameters(), lr=tc.learning_rate,
        weight_decay=tc.weight_decay, betas=(0.9, 0.95),
    )

    use_amp = device.type == "cuda"
    scaler = torch.amp.GradScaler("cuda") if use_amp else None

    os.makedirs(tc.output_dir, exist_ok=True)
    with open(os.path.join(tc.output_dir, "config.json"), "w") as f:
        json.dump({"model": vars(mc), "train": vars(tc)}, f, indent=2)

    model.train()
    step, best_eval = 0, float("inf")
    losses = []
    t0 = time.time()

    print(f"\nTraining for {tc.max_steps} steps (early_exit={mc.early_exit})...")
    print(f"{'Step':>6} | {'LR':>10} | {'Train':>10} | {'Eval':>10} | {'AvgExit':>8} | {'Time':>8}")
    print("-" * 72)

    while step < tc.max_steps:
        for x, y in train_loader:
            if step >= tc.max_steps:
                break

            x, y = x.to(device), y.to(device)
            lr = get_lr(step, tc)
            for pg in optimizer.param_groups:
                pg["lr"] = lr

            if use_amp:
                with torch.amp.autocast("cuda"):
                    _, loss = model(x, y)
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), tc.grad_clip)
                scaler.step(optimizer)
                scaler.update()
            else:
                _, loss = model(x, y)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), tc.grad_clip)
                optimizer.step()

            optimizer.zero_grad(set_to_none=True)
            losses.append(loss.item())

            if step % 100 == 0:
                avg = sum(losses[-100:]) / len(losses[-100:])
                elapsed = time.time() - t0
                print(f"{step:6d} | {lr:10.6f} | {avg:10.4f} | {'--':>10} | {'--':>8} | {elapsed:7.1f}s")

            if step > 0 and step % tc.eval_interval == 0:
                el, avg_exit = evaluate(model, eval_loader, device)
                avg_train = sum(losses[-tc.eval_interval:]) / min(len(losses), tc.eval_interval)
                elapsed = time.time() - t0
                exit_str = f"{avg_exit:.1f}/{mc.n_layers}" if mc.early_exit else "off"
                print(f"{step:6d} | {lr:10.6f} | {avg_train:10.4f} | {el:10.4f} | {exit_str:>8} | {elapsed:7.1f}s")

                if el < best_eval:
                    best_eval = el
                    torch.save({
                        "step": step,
                        "model_state_dict": model.state_dict(),
                        "config": vars(mc),
                        "eval_loss": el,
                    }, os.path.join(tc.output_dir, "best_model.pt"))
                    print(f"  -> Best model (eval={el:.4f}, avg_exit={exit_str})")

            if step > 0 and step % tc.save_interval == 0:
                torch.save({
                    "step": step,
                    "model_state_dict": model.state_dict(),
                    "config": vars(mc),
                }, os.path.join(tc.output_dir, f"step_{step}.pt"))

            step += 1

    # Final save
    torch.save({
        "step": step,
        "model_state_dict": model.state_dict(),
        "config": vars(mc),
        "train_losses": losses,
    }, os.path.join(tc.output_dir, "final_model.pt"))

    elapsed = time.time() - t0
    print(f"\nDone! {elapsed:.0f}s, best eval: {best_eval:.4f}")


if __name__ == "__main__":
    train()


### `json_repair.py`

In [ ]:
%%writefile json_repair.py
"""JSON repair and schema enforcement for GobyLLM tool calls."""

import json
import re
from difflib import SequenceMatcher


# ── JSON repair ─────────────────────────────────────────────────────────


def repair_json(s: str) -> str:
    """Attempt to fix broken JSON string into valid JSON."""

    if not s or not s.strip():
        return s

    s = s.strip()

    # Remove any leading/trailing text that isn't part of the JSON
    # Find the first { and work from there
    start = s.find("{")
    if start < 0:
        return s
    s = s[start:]

    # Single quotes → double quotes (but not inside already-double-quoted strings)
    # Simple approach: replace all single quotes with double quotes,
    # then fix cases where we broke things
    s = _fix_quotes(s)

    # Remove trailing commas before } or ]
    s = re.sub(r",\s*([}\]])", r"\1", s)

    # Fix unquoted keys: {name: "foo"} → {"name": "foo"}
    s = re.sub(r'(?<=[{,])\s*([a-zA-Z_][a-zA-Z0-9_]*)\s*:', r' "\1":', s)

    # Close unclosed braces/brackets
    s = _close_brackets(s)

    # Try to parse — if it works, return cleaned version
    try:
        obj = json.loads(s)
        return json.dumps(obj)
    except json.JSONDecodeError:
        pass

    # More aggressive: try to extract just the tool call pattern
    m = re.search(r'\{\s*"name"\s*:\s*"([^"]+)"\s*,\s*"arguments"\s*:\s*(\{.*)', s, re.DOTALL)
    if m:
        name = m.group(1)
        args_str = _close_brackets(m.group(2))
        try:
            args = json.loads(args_str)
            return json.dumps({"name": name, "arguments": args})
        except json.JSONDecodeError:
            # Try to parse args more aggressively
            args = _extract_kv_pairs(args_str)
            if args:
                return json.dumps({"name": name, "arguments": args})

    return s


def _fix_quotes(s: str) -> str:
    """Replace single quotes with double quotes, handling escapes."""
    result = []
    in_double = False
    in_single = False
    i = 0
    while i < len(s):
        c = s[i]
        if c == "\\" and i + 1 < len(s):
            result.append(c)
            result.append(s[i + 1])
            i += 2
            continue
        if c == '"' and not in_single:
            in_double = not in_double
            result.append(c)
        elif c == "'" and not in_double:
            in_single = not in_single
            result.append('"')  # replace single with double
        else:
            result.append(c)
        i += 1
    return "".join(result)


def _close_brackets(s: str) -> str:
    """Close any unclosed { or [ brackets."""
    stack = []
    in_string = False
    escape = False
    for c in s:
        if escape:
            escape = False
            continue
        if c == "\\":
            escape = True
            continue
        if c == '"':
            in_string = not in_string
            continue
        if in_string:
            continue
        if c == "{":
            stack.append("}")
        elif c == "[":
            stack.append("]")
        elif c in "}]":
            if stack and stack[-1] == c:
                stack.pop()
    # Close any remaining open brackets
    while stack:
        s += stack.pop()
    return s


def _extract_kv_pairs(s: str) -> dict:
    """Last-resort extraction of key-value pairs from messy text."""
    pairs = {}
    # Match "key": "value" or "key": number
    for m in re.finditer(r'"([^"]+)"\s*:\s*("([^"]*)"|([\d.]+)|(true|false|null))', s):
        key = m.group(1)
        if m.group(3) is not None:
            pairs[key] = m.group(3)
        elif m.group(4) is not None:
            try:
                pairs[key] = float(m.group(4)) if "." in m.group(4) else int(m.group(4))
            except ValueError:
                pairs[key] = m.group(4)
        elif m.group(5) is not None:
            pairs[key] = {"true": True, "false": False, "null": None}[m.group(5)]
    return pairs


# ── Tool schema matching ────────────────────────────────────────────────


def fuzzy_match_tool(name: str, tool_names: list, threshold: float = 0.6) -> str:
    """Find closest matching tool name. Returns original if no good match."""
    if name in tool_names:
        return name

    best_match = None
    best_score = 0
    for tn in tool_names:
        score = SequenceMatcher(None, name.lower(), tn.lower()).ratio()
        if score > best_score:
            best_score = score
            best_match = tn

    return best_match if best_score >= threshold else name


def validate_and_fix(tc_dict: dict, tools: list) -> dict:
    """Validate a tool call against tool schemas and fix issues."""

    if not tools or not tc_dict:
        return tc_dict

    name = tc_dict.get("name", "")
    args = tc_dict.get("arguments", {})
    if isinstance(args, str):
        try:
            args = json.loads(args)
        except json.JSONDecodeError:
            args = {}

    # Find matching tool schema
    tool_names = [t["function"]["name"] for t in tools if "function" in t]
    matched_name = fuzzy_match_tool(name, tool_names)

    # Get schema for matched tool
    schema = None
    for t in tools:
        if t.get("function", {}).get("name") == matched_name:
            schema = t["function"].get("parameters", {})
            break

    if schema:
        properties = schema.get("properties", {})
        required = schema.get("required", [])

        # Coerce types
        fixed_args = {}
        for key, prop in properties.items():
            if key in args:
                fixed_args[key] = _coerce_type(args[key], prop)
            elif key in required:
                # Fill default for missing required param
                fixed_args[key] = _default_for_type(prop)

        # Keep any extra args the model provided (might be useful)
        for key in args:
            if key not in fixed_args:
                fixed_args[key] = args[key]

        args = fixed_args

    return {"name": matched_name, "arguments": args}


def _coerce_type(value, prop: dict):
    """Coerce a value to match the expected JSON schema type."""
    expected = prop.get("type", "string")
    enum = prop.get("enum")

    if expected == "number":
        try:
            return float(value)
        except (ValueError, TypeError):
            return 0.0

    if expected == "integer":
        try:
            return int(float(value))
        except (ValueError, TypeError):
            return 0

    if expected == "boolean":
        if isinstance(value, bool):
            return value
        return str(value).lower() in ("true", "1", "yes", "on")

    # String type
    value = str(value)

    # Fuzzy match against enum values
    if enum and value not in enum:
        best = None
        best_score = 0
        for e in enum:
            score = SequenceMatcher(None, value.lower(), e.lower()).ratio()
            if score > best_score:
                best_score = score
                best = e
        if best and best_score >= 0.5:
            return best

    return value


def _default_for_type(prop: dict):
    """Generate a sensible default for a missing required parameter."""
    t = prop.get("type", "string")
    enum = prop.get("enum")
    if enum:
        return enum[0]
    if t == "number":
        return 0.0
    if t == "integer":
        return 0
    if t == "boolean":
        return False
    return ""


# ── Extraction ──────────────────────────────────────────────────────────


def extract_tool_calls(text: str, tools: list = None) -> list:
    """Extract, repair, and validate all tool calls from model output text."""

    import uuid
    results = []
    raw_tcs = []

    # Method 1: <tool_call> tags
    for m in re.finditer(r"<tool_call>(.*?)</tool_call>", text, re.DOTALL):
        raw_tcs.append(m.group(1).strip())

    # Method 2: raw JSON with name+arguments
    if not raw_tcs:
        for m in re.finditer(
            r'\{[^{}]*"name"\s*:\s*"[^"]*"[^{}]*"arguments"\s*:\s*\{[^}]*\}[^}]*\}', text
        ):
            raw_tcs.append(m.group(0))

    # Method 3: even looser — just find {"name": "something" and grab what follows
    if not raw_tcs:
        for m in re.finditer(r'\{\s*"name"\s*:\s*"([^"]+)".*?(?:\}|$)', text, re.DOTALL):
            raw_tcs.append(m.group(0))

    for raw in raw_tcs:
        repaired = repair_json(raw)
        try:
            tc = json.loads(repaired)
        except json.JSONDecodeError:
            continue

        if "name" not in tc:
            continue

        # Validate against schema
        if tools:
            tc = validate_and_fix(tc, tools)

        results.append({
            "id": f"call_{uuid.uuid4().hex[:8]}",
            "type": "function",
            "function": {
                "name": tc["name"],
                "arguments": json.dumps(tc.get("arguments", {})),
            },
        })

    return results


def clean_response_text(text: str) -> str:
    """Remove tool call artifacts from response text."""
    text = re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL)
    text = re.sub(r"<tool_call>.*?</tool_call>", "", text, flags=re.DOTALL)
    text = re.sub(r'\{"name"\s*:.*?"arguments"\s*:\s*\{[^}]*\}\s*\}', "", text)
    text = text.replace("<|im_end|>", "").strip()
    return text


if __name__ == "__main__":
    # Test cases
    tests = [
        # Broken quotes
        ("{'name': 'turn_on', 'arguments': {'device': 'kitchen lights'}}", None),
        # Trailing comma
        ('{"name": "set_temperature", "arguments": {"temperature": 22, "mode": "cool",}}', None),
        # Unclosed brace (truncated)
        ('{"name": "lock_door", "arguments": {"door": "front door"', None),
        # Unquoted keys
        ('{name: "turn_off", arguments: {device: "fan"}}', None),
        # Typo in tool name
        ('{"name": "trun_on", "arguments": {"device": "lights"}}',
         [{"type": "function", "function": {"name": "turn_on", "parameters": {"type": "object", "properties": {"device": {"type": "string"}}, "required": ["device"]}}}]),
        # Wrong type
        ('{"name": "set_brightness", "arguments": {"device": "lamp", "level": "50"}}',
         [{"type": "function", "function": {"name": "set_brightness", "parameters": {"type": "object", "properties": {"device": {"type": "string"}, "level": {"type": "integer"}}, "required": ["device", "level"]}}}]),
    ]

    for raw, tools in tests:
        repaired = repair_json(raw)
        try:
            parsed = json.loads(repaired)
            if tools:
                parsed = validate_and_fix(parsed, tools)
            print(f"  IN:  {raw[:70]}")
            print(f"  OUT: {json.dumps(parsed)}")
            print()
        except json.JSONDecodeError as e:
            print(f"  IN:  {raw[:70]}")
            print(f"  ERR: {e}")
            print()


### `inference.py`

In [ ]:
%%writefile inference.py
"""GobyLLM inference -- OpenAI-compatible with early exit reporting."""

import json
import re
import time
import uuid

import torch
from tokenizers import Tokenizer

from config import GobyConfig
from model import GobyLLM


class GobyInference:
    def __init__(self, checkpoint_path, tokenizer_path, device="cpu"):
        self.device = torch.device(device)
        self.tokenizer = Tokenizer.from_file(tokenizer_path)

        ckpt = torch.load(checkpoint_path, map_location=self.device, weights_only=False)
        self.config = GobyConfig(**ckpt["config"])
        self.model = GobyLLM(self.config).to(self.device)
        self.model.load_state_dict(ckpt["model_state_dict"])
        self.model.eval()

        total, router = self.model.param_count()
        print(f"GobyLLM loaded: {total/1e6:.1f}M params, early_exit={self.config.early_exit}")
        if self.config.early_exit:
            print(f"  Router overhead: {router:,} params ({router/total*100:.2f}%)")
            print(f"  Exit threshold: {self.config.exit_threshold}, min_layer: {self.config.min_exit_layer}")

    def chat_completion(self, messages, tools=None, tool_choice="auto",
                        temperature=0.7, max_tokens=256, top_k=50,
                        model=None, exit_threshold=None, **kwargs):
        """OpenAI-compatible chat completion."""

        model_name = model or "goby-25m"

        effective_tools = tools
        if tool_choice == "none":
            effective_tools = None

        self._last_tools = effective_tools
        prompt = self._format_prompt(messages, effective_tools)
        input_ids = self.tokenizer.encode(prompt).ids
        prompt_tokens = len(input_ids)
        input_t = torch.tensor([input_ids], dtype=torch.long, device=self.device)

        output_t, exit_layers = self.model.generate(
            input_t, max_tokens, temperature, top_k,
            exit_threshold=exit_threshold,
        )
        output_text = self.tokenizer.decode(output_t[0].tolist()[prompt_tokens:])

        resp = self._build_response(output_text, exit_layers, prompt_tokens, model_name)

        # Enforce tool_choice
        msg = resp["choices"][0]["message"]
        if tool_choice == "none":
            msg.pop("tool_calls", None)
            resp["choices"][0]["finish_reason"] = "stop"
        elif isinstance(tool_choice, dict):
            forced = tool_choice.get("function", {}).get("name")
            if forced and msg.get("tool_calls"):
                for tc in msg["tool_calls"]:
                    tc["function"]["name"] = forced

        return resp

    def _format_prompt(self, messages, tools=None):
        parts = []
        has_system = False
        for msg in messages:
            role = msg.get("role", "user")
            content = msg.get("content") or ""
            if role == "tool":
                tool_call_id = msg.get("tool_call_id", "")
                parts.append(f"<|im_start|>tool\n[{tool_call_id}] {content}<|im_end|>")
                continue
            if role == "system":
                has_system = True
                if tools:
                    content += f"\n\n# Tools\n{json.dumps(tools, separators=(',', ':'))}"
            parts.append(f"<|im_start|>{role}\n{content}<|im_end|>")
        if not has_system and tools:
            sys = "You are a helpful assistant.\n\n# Tools\n" + json.dumps(tools, separators=(",", ":"))
            parts.insert(0, f"<|im_start|>system\n{sys}<|im_end|>")
        parts.append("<|im_start|>assistant\n")
        return "\n".join(parts)

    def _build_response(self, text, exit_layers, prompt_tokens=0, model_name="goby-25m"):
        from json_repair import extract_tool_calls, clean_response_text

        thinking = None
        m = re.search(r"<think>(.*?)</think>", text, re.DOTALL)
        if m:
            thinking = m.group(1).strip()

        tool_calls = extract_tool_calls(text, tools=self._last_tools)
        resp_text = clean_response_text(text)

        message = {"role": "assistant", "content": resp_text if resp_text else None}
        if tool_calls:
            message["tool_calls"] = tool_calls
            if not resp_text:
                message["content"] = None

        completion_tokens = len(exit_layers)
        avg_exit = sum(exit_layers) / max(len(exit_layers), 1)
        result = {
            "id": f"chatcmpl-{uuid.uuid4().hex[:24]}",
            "object": "chat.completion",
            "created": int(time.time()),
            "model": model_name,
            "system_fingerprint": f"goby_{self.config.n_layers}l_{self.config.d_model}d",
            "choices": [{
                "index": 0,
                "message": message,
                "finish_reason": "tool_calls" if tool_calls else "stop",
                "logprobs": None,
            }],
            "usage": {
                "prompt_tokens": prompt_tokens,
                "completion_tokens": completion_tokens,
                "total_tokens": prompt_tokens + completion_tokens,
            },
        }
        if thinking:
            result["_thinking"] = thinking
        result["_early_exit"] = {
            "avg_layer": round(avg_exit, 1),
            "max_layers": self.config.n_layers,
            "compute_saved": f"{(1 - avg_exit / self.config.n_layers) * 100:.0f}%",
        }
        return result

    @torch.no_grad()
    def benchmark(self, seq_len=128, n_runs=100):
        """Compare full model vs early exit inference speed."""

        dummy = torch.randint(0, self.config.vocab_size, (1, seq_len),
                              dtype=torch.long, device=self.device)

        # Warmup
        for _ in range(5):
            self.model(dummy)

        # Full model (no early exit)
        old_ee = self.config.early_exit
        self.config.early_exit = False
        times_full = []
        for _ in range(n_runs):
            s = time.perf_counter()
            self.model(dummy)
            times_full.append(time.perf_counter() - s)

        # With early exit
        self.config.early_exit = old_ee
        times_ee = []
        exit_layer_sum = 0
        for _ in range(n_runs):
            prompt = torch.randint(0, self.config.vocab_size, (1, 16), device=self.device)
            s = time.perf_counter()
            _, exits = self.model.generate(prompt, max_new_tokens=1)
            times_ee.append(time.perf_counter() - s)
            exit_layer_sum += exits[0]

        avg_full = sum(times_full) / len(times_full) * 1000
        avg_ee = sum(times_ee) / len(times_ee) * 1000
        avg_exit = exit_layer_sum / n_runs

        print(f"\nBenchmark ({self.device}, {n_runs} runs):")
        print(f"  Full model:  {avg_full:.2f}ms/forward")
        print(f"  Early exit:  {avg_ee:.2f}ms/generate (avg layer {avg_exit:.1f}/{self.config.n_layers})")
        if avg_full > 0:
            print(f"  Potential speedup: {avg_full/avg_ee:.1f}x (on untrained model — improves after training)")


# ── HTTP server ─────────────────────────────────────────────────────────


def run_server(engine, host="0.0.0.0", port=8000):
    from http.server import HTTPServer, BaseHTTPRequestHandler

    class H(BaseHTTPRequestHandler):
        def do_POST(self):
            if self.path == "/v1/chat/completions":
                body = json.loads(self.rfile.read(int(self.headers.get("Content-Length", 0))))
                result = engine.chat_completion(
                    messages=body.get("messages", []),
                    tools=body.get("tools"),
                    temperature=body.get("temperature", 0.7),
                    max_tokens=body.get("max_tokens", 256),
                )
                self.send_response(200)
                self.send_header("Content-Type", "application/json")
                self.end_headers()
                self.wfile.write(json.dumps(result).encode())
            else:
                self.send_response(404)
                self.end_headers()

        def do_GET(self):
            if self.path == "/v1/models":
                self.send_response(200)
                self.send_header("Content-Type", "application/json")
                self.end_headers()
                self.wfile.write(json.dumps({
                    "data": [{"id": "goby-25m", "object": "model",
                              "meta": {"early_exit": True, "params": "25.5M"}}]
                }).encode())
            else:
                self.send_response(404)
                self.end_headers()

        def log_message(self, fmt, *args):
            print(f"[{self.log_date_time_string()}] {fmt % args}")

    srv = HTTPServer((host, port), H)
    print(f"GobyLLM server at http://{host}:{port}")
    print(f'  client = OpenAI(base_url="http://localhost:{port}/v1", api_key="unused")')
    srv.serve_forever()


# ── CLI ─────────────────────────────────────────────────────────────────


def main():
    import argparse
    p = argparse.ArgumentParser()
    p.add_argument("--checkpoint", default="checkpoints/best_model.pt")
    p.add_argument("--tokenizer", default="data/tokenizer.json")
    p.add_argument("--device", default="cpu")
    p.add_argument("--serve", action="store_true")
    p.add_argument("--port", type=int, default=8000)
    p.add_argument("--benchmark", action="store_true")
    p.add_argument("--interactive", action="store_true")
    args = p.parse_args()

    engine = GobyInference(args.checkpoint, args.tokenizer, args.device)

    if args.benchmark:
        engine.benchmark()
    elif args.serve:
        run_server(engine, port=args.port)
    elif args.interactive:
        tools = [
            {"type": "function", "function": {"name": "turn_on", "description": "Turn on a device",
             "parameters": {"type": "object", "properties": {"device": {"type": "string"}}, "required": ["device"]}}},
            {"type": "function", "function": {"name": "set_temperature", "description": "Set thermostat",
             "parameters": {"type": "object", "properties": {
                 "temperature": {"type": "number"}, "mode": {"type": "string", "enum": ["heat","cool","auto"]}
             }, "required": ["temperature", "mode"]}}},
        ]
        print("\nGobyLLM Interactive (type 'quit' to exit)")
        msgs = [{"role": "system", "content": "You are a helpful assistant."}]
        while True:
            inp = input("\nYou> ").strip()
            if inp.lower() in ("quit", "exit", "q"):
                break
            msgs.append({"role": "user", "content": inp})
            result = engine.chat_completion(msgs, tools=tools)
            ch = result["choices"][0]
            ee = result.get("_early_exit", {})
            if ch.get("_thinking"):
                print(f"[Think] {ch['_thinking']}")
            msg = ch["message"]
            if msg.get("tool_calls"):
                for tc in msg["tool_calls"]:
                    print(f"[Tool] {tc['function']['name']}({tc['function']['arguments']})")
            if msg.get("content"):
                print(f"Goby> {msg['content']}")
            if ee:
                print(f"  [exit layer {ee['avg_layer']}/{ee['max_layers']}, {ee['compute_saved']} saved]")
            msgs.append(msg)


if __name__ == "__main__":
    main()


### `rpi_runner.py`

In [ ]:
%%writefile rpi_runner.py
"""GobyLLM RPi Runner -- optimized inference for Raspberry Pi and edge devices."""

import argparse
import json
import math
import re
import time
import uuid

import torch
import torch.nn as nn
import torch.nn.functional as F
from tokenizers import Tokenizer

from config import GobyConfig
from model import GobyLLM, apply_rope


class KVCache:
    """Pre-allocated KV cache for a single attention layer."""
    def __init__(self, max_seq_len, n_kv_heads, head_dim, device, dtype):
        self.max_seq_len = max_seq_len
        self.k = torch.zeros(1, n_kv_heads, max_seq_len, head_dim, device=device, dtype=dtype)
        self.v = torch.zeros(1, n_kv_heads, max_seq_len, head_dim, device=device, dtype=dtype)
        self.pos = 0

    def update(self, k_new, v_new):
        """Append new KV to cache. k_new/v_new: [1, n_kv_heads, T, head_dim]"""
        T = k_new.shape[2]
        end = self.pos + T
        self.k[:, :, self.pos:end, :] = k_new
        self.v[:, :, self.pos:end, :] = v_new
        self.pos = end

    def get(self):
        """Return cached K, V up to current position."""
        return self.k[:, :, :self.pos, :], self.v[:, :, :self.pos, :]

    def reset(self):
        self.pos = 0


class GobyRunner:
    """Optimized GobyLLM inference engine for edge devices."""

    def __init__(self, checkpoint_path, tokenizer_path, quantize=True, device="cpu"):
        self.device = torch.device(device)
        self.tokenizer = Tokenizer.from_file(tokenizer_path)

        # Load model
        ckpt = torch.load(checkpoint_path, map_location=self.device, weights_only=False)
        self.config = GobyConfig(**ckpt["config"])
        self.model = GobyLLM(self.config).to(self.device)
        self.model.load_state_dict(ckpt["model_state_dict"])
        self.model.eval()

        total, router = self.model.param_count()
        fp32_mb = total * 4 / 1e6
        print(f"GobyLLM loaded: {total/1e6:.1f}M params ({fp32_mb:.0f}MB FP32)")

        # INT8 quantization
        self.quantized = False
        if quantize and device == "cpu":
            self._quantize()

        # Pre-extract model components for fast access
        self.tok_emb = self.model.tok_emb
        self.blocks = self.model.blocks
        self.norm = self.model.norm
        self.lm_head = self.model.lm_head
        self.rope_cos = self.model.rope_cos
        self.rope_sin = self.model.rope_sin
        self.exit_routers = self.model.exit_routers if self.config.early_exit else None
        self.n_layers = self.config.n_layers
        self.head_dim = self.config.d_model // self.config.n_heads

        # Pre-allocate KV caches
        self.kv_caches = self._alloc_caches()

        # Warmup
        self._warmup()

    def _quantize(self):
        """Apply INT8 dynamic quantization."""

        # Untie weights before quantization (shared params confuse quantizer)
        self.model.lm_head = nn.Linear(
            self.config.d_model, self.config.vocab_size, bias=False
        )
        self.model.lm_head.weight = nn.Parameter(self.model.tok_emb.weight.clone())

        self.model = torch.quantization.quantize_dynamic(
            self.model, {nn.Linear}, dtype=torch.qint8
        )
        self.quantized = True

        # Estimate size
        param_bytes = sum(
            p.nelement() * (1 if p.dtype == torch.qint8 else p.element_size())
            for p in self.model.parameters()
        )
        print(f"INT8 quantized: ~{param_bytes/1e6:.0f}MB")

    def _alloc_caches(self):
        """Pre-allocate KV caches for all layers."""
        dtype = torch.float32
        return [
            KVCache(
                self.config.max_seq_len, self.config.n_kv_heads,
                self.head_dim, self.device, dtype
            )
            for _ in range(self.n_layers)
        ]

    def _reset_caches(self):
        for c in self.kv_caches:
            c.reset()

    def _warmup(self):
        """Run a dummy forward pass to trigger lazy initializations."""
        dummy = torch.tensor([[1, 2, 3]], dtype=torch.long, device=self.device)
        self._reset_caches()
        self._forward_prompt(dummy)
        self._reset_caches()
        print("Warmup done")

    # ── Cached forward passes ───────────────────────────────────────────

    def _attention_cached(self, attn, x, rope_cos, rope_sin, kv_cache):
        """Attention forward with KV cache update."""
        B, T, _ = x.shape

        q = attn.wq(x).view(B, T, attn.n_heads, self.head_dim).transpose(1, 2)
        k = attn.wk(x).view(B, T, attn.n_kv_heads, self.head_dim).transpose(1, 2)
        v = attn.wv(x).view(B, T, attn.n_kv_heads, self.head_dim).transpose(1, 2)

        q = apply_rope(q, rope_cos, rope_sin)
        k = apply_rope(k, rope_cos, rope_sin)

        # Update cache
        kv_cache.update(k, v)
        k_full, v_full = kv_cache.get()

        # Expand KV heads for GQA
        if attn.n_rep > 1:
            k_full = k_full.repeat_interleave(attn.n_rep, dim=1)
            v_full = v_full.repeat_interleave(attn.n_rep, dim=1)

        # Attention
        scale = math.sqrt(self.head_dim)
        scores = (q @ k_full.transpose(-2, -1)) / scale

        # Causal mask only needed for prompt (T > 1)
        if T > 1:
            kv_len = k_full.shape[2]
            mask = torch.tril(torch.ones(T, kv_len, device=x.device))
            # Offset: if cache had prev tokens, mask should allow attending to them
            # Since we only call T>1 for prompt (cache starts empty), standard causal is fine
            scores = scores.masked_fill(mask.unsqueeze(0).unsqueeze(0) == 0, float("-inf"))

        attn_weights = F.softmax(scores, dim=-1)
        out = (attn_weights @ v_full).transpose(1, 2).contiguous().view(B, T, -1)
        return attn.wo(out)

    def _block_cached(self, block, x, rope_cos, rope_sin, kv_cache):
        """Single block forward with KV cache (supports parallel residual)."""
        if block.parallel:
            h = block.norm(x)
            attn_out = self._attention_cached(block.attn, h, rope_cos, rope_sin, kv_cache)
            ffn_out = block.ffn(h)
            return x + attn_out + ffn_out
        else:
            h = block.norm(x)
            attn_out = self._attention_cached(block.attn, h, rope_cos, rope_sin, kv_cache)
            x = x + attn_out
            return x + block.ffn(block.norm2(x))

    def _forward_prompt(self, token_ids, max_depth=None):
        """Process prompt through layers with KV caching."""

        if max_depth is None:
            max_depth = self.n_layers

        B, T = token_ids.shape
        x = self.tok_emb(token_ids)
        rope_cos = self.rope_cos[:T]
        rope_sin = self.rope_sin[:T]

        exit_depth = self.n_layers
        for i in range(self.n_layers):
            x = self._block_cached(self.blocks[i], x, rope_cos, rope_sin, self.kv_caches[i])

            # Check router for early exit depth determination
            if (self.config.early_exit and self.exit_routers is not None
                    and i >= self.config.min_exit_layer
                    and i < self.n_layers - 1):
                conf = torch.sigmoid(self.exit_routers[i](x[:, -1:, :].mean(dim=1)))
                if conf.item() > self.config.exit_threshold and exit_depth == self.n_layers:
                    exit_depth = i + 1
                    # Don't break — we still need to fill ALL KV caches for flexibility
                    # But we record the depth for generation

        return x, exit_depth

    def _forward_one_token(self, token_id, pos, depth):
        """Process a single new token through `depth` layers using KV cache."""

        x = self.tok_emb(token_id)  # [1, 1, d_model]
        rope_cos = self.rope_cos[pos:pos + 1]
        rope_sin = self.rope_sin[pos:pos + 1]

        for i in range(depth):
            x = self._block_cached(self.blocks[i], x, rope_cos, rope_sin, self.kv_caches[i])

        logits = self.lm_head(self.norm(x))  # [1, 1, vocab]
        return logits[:, -1, :]  # [1, vocab]

    # ── Generation ──────────────────────────────────────────────────────

    @torch.no_grad()
    def generate(self, prompt_ids, max_tokens=256, temperature=0.7, top_k=50):
        """KV-cached generation with fixed-depth early exit."""

        self._reset_caches()

        prompt_ids = prompt_ids.to(self.device)
        B, prompt_len = prompt_ids.shape

        # Phase 1: prompt processing (all layers, to fill KV cache)
        t_prompt_start = time.perf_counter()
        hidden, exit_depth = self._forward_prompt(prompt_ids)
        logits = self.lm_head(self.norm(hidden))[:, -1, :]
        t_prompt = time.perf_counter() - t_prompt_start

        # Phase 2: token-by-token generation using KV cache + fixed depth
        generated = []
        t_gen_start = time.perf_counter()

        for step in range(max_tokens):
            logits_scaled = logits / temperature
            if top_k > 0:
                v, _ = torch.topk(logits_scaled, min(top_k, logits_scaled.size(-1)))
                logits_scaled[logits_scaled < v[:, [-1]]] = float("-inf")

            probs = F.softmax(logits_scaled, dim=-1)
            next_id = torch.multinomial(probs, num_samples=1)  # [1, 1]
            generated.append(next_id.item())

            if next_id.item() == self.config.eos_id:
                break

            pos = prompt_len + step
            if pos >= self.config.max_seq_len - 1:
                break

            logits = self._forward_one_token(next_id, pos, exit_depth)

        t_gen = time.perf_counter() - t_gen_start
        n_gen = len(generated)

        output_ids = torch.cat([
            prompt_ids,
            torch.tensor([generated], dtype=torch.long, device=self.device)
        ], dim=1)

        meta = {
            "prompt_tokens": prompt_len,
            "generated_tokens": n_gen,
            "exit_depth": exit_depth,
            "max_depth": self.n_layers,
            "compute_saved": f"{(1 - exit_depth / self.n_layers) * 100:.0f}%",
            "prompt_ms": round(t_prompt * 1000, 1),
            "gen_ms": round(t_gen * 1000, 1),
            "tokens_per_sec": round(n_gen / t_gen, 1) if t_gen > 0 else 0,
        }

        return output_ids, meta

    # ── OpenAI-compatible API ───────────────────────────────────────────

    def chat_completion(self, messages, tools=None, tool_choice="auto",
                        temperature=0.7, max_tokens=256, top_k=50,
                        model=None, n=1, stop=None, stream=False,
                        **kwargs):
        """OpenAI-compatible chat completion."""

        model_name = model or "goby-25m"

        # Handle tool_choice
        effective_tools = tools
        forced_tool = None
        if tool_choice == "none":
            effective_tools = None  # hide tools from model
        elif isinstance(tool_choice, dict):
            # Force a specific tool
            forced_tool = tool_choice.get("function", {}).get("name")

        self._last_tools = effective_tools
        prompt = self._format_prompt(messages, effective_tools)
        input_ids = self.tokenizer.encode(prompt).ids
        prompt_tokens = len(input_ids)
        input_t = torch.tensor([input_ids], dtype=torch.long, device=self.device)

        output_t, meta = self.generate(input_t, max_tokens, temperature, top_k)
        output_text = self.tokenizer.decode(output_t[0].tolist()[prompt_tokens:])

        resp = self._build_response(output_text, meta, model_name, prompt_tokens)

        # Enforce tool_choice constraints
        msg = resp["choices"][0]["message"]
        if tool_choice == "none":
            msg.pop("tool_calls", None)
            resp["choices"][0]["finish_reason"] = "stop"
        elif tool_choice == "required" and not msg.get("tool_calls"):
            # Model didn't call a tool but was required to — mark as failure
            resp["choices"][0]["finish_reason"] = "stop"
        elif forced_tool and msg.get("tool_calls"):
            # Override tool name to the forced one
            for tc in msg["tool_calls"]:
                tc["function"]["name"] = forced_tool
            resp["choices"][0]["finish_reason"] = "tool_calls"

        return resp

    def _format_prompt(self, messages, tools=None):
        parts = []
        has_system = False
        for msg in messages:
            role = msg.get("role", "user")
            content = msg.get("content") or ""

            # Handle tool results in conversation
            if role == "tool":
                tool_call_id = msg.get("tool_call_id", "")
                parts.append(f"<|im_start|>tool\n[{tool_call_id}] {content}<|im_end|>")
                continue

            # Inject tools into system prompt
            if role == "system":
                has_system = True
                if tools:
                    content += f"\n\n# Tools\n{json.dumps(tools, separators=(',', ':'))}"

            parts.append(f"<|im_start|>{role}\n{content}<|im_end|>")

        # If no system message was provided, add one with tools
        if not has_system and tools:
            sys_content = "You are a helpful assistant.\n\n# Tools\n" + json.dumps(tools, separators=(",", ":"))
            parts.insert(0, f"<|im_start|>system\n{sys_content}<|im_end|>")

        parts.append("<|im_start|>assistant\n")
        return "\n".join(parts)

    def _build_response(self, text, meta, model_name="goby-25m",
                        prompt_tokens=0, stream=False):
        from json_repair import extract_tool_calls, clean_response_text

        thinking = None
        m = re.search(r"<think>(.*?)</think>", text, re.DOTALL)
        if m:
            thinking = m.group(1).strip()

        tool_calls = extract_tool_calls(text, tools=getattr(self, '_last_tools', None))
        resp_text = clean_response_text(text)

        # OpenAI spec: content must be null when tool_calls are present, not empty string
        message = {"role": "assistant", "content": resp_text if resp_text else None}
        if tool_calls:
            message["tool_calls"] = tool_calls
            if not resp_text:
                message["content"] = None

        finish_reason = "tool_calls" if tool_calls else "stop"
        completion_tokens = meta.get("tokens_generated", 0)

        resp = {
            "id": f"chatcmpl-{uuid.uuid4().hex[:24]}",
            "object": "chat.completion",
            "created": int(time.time()),
            "model": model_name,
            "system_fingerprint": f"goby_{self.config.n_layers}l_{self.config.d_model}d",
            "choices": [{
                "index": 0,
                "message": message,
                "finish_reason": finish_reason,
                "logprobs": None,
            }],
            "usage": {
                "prompt_tokens": prompt_tokens,
                "completion_tokens": completion_tokens,
                "total_tokens": prompt_tokens + completion_tokens,
            },
        }

        # Non-standard extensions (prefixed with underscore)
        if thinking:
            resp["_thinking"] = thinking
        if meta:
            resp["_performance"] = meta

        return resp

    # ── Benchmarking ────────────────────────────────────────────────────

    def benchmark(self, prompt_len=32, gen_tokens=64, n_runs=20):
        """Benchmark cached generation vs naive (recompute-all) generation."""

        print(f"\n{'='*60}")
        print(f"GobyLLM Benchmark ({'INT8' if self.quantized else 'FP32'}, {self.device})")
        print(f"  Prompt: {prompt_len} tokens, Generate: {gen_tokens} tokens")
        print(f"{'='*60}")

        prompt = torch.randint(1, self.config.vocab_size, (1, prompt_len), device=self.device)

        # Warm up
        for _ in range(3):
            self.generate(prompt, max_tokens=8)

        # Cached generation (this runner)
        times_cached = []
        depths = []
        for _ in range(n_runs):
            s = time.perf_counter()
            _, meta = self.generate(prompt, max_tokens=gen_tokens, temperature=0.8)
            times_cached.append(time.perf_counter() - s)
            depths.append(meta["exit_depth"])

        avg_cached = sum(times_cached) / len(times_cached) * 1000
        avg_depth = sum(depths) / len(depths)
        tok_per_sec = gen_tokens / (avg_cached / 1000)

        # Naive generation (no KV cache, full recompute — like inference.py)
        times_naive = []
        for _ in range(min(n_runs, 5)):  # fewer runs since it's slow
            s = time.perf_counter()
            idx = prompt.clone()
            for _ in range(gen_tokens):
                idx_cond = idx[:, -self.config.max_seq_len:]
                with torch.no_grad():
                    logits, _ = self.model(idx_cond)
                next_id = logits[:, -1, :].argmax(dim=-1, keepdim=True)
                idx = torch.cat([idx, next_id], dim=1)
            times_naive.append(time.perf_counter() - s)

        avg_naive = sum(times_naive) / len(times_naive) * 1000

        print(f"\n  Naive (no cache):     {avg_naive:8.1f}ms  ({gen_tokens*1000/avg_naive:5.1f} tok/s)")
        print(f"  KV-cached + early exit: {avg_cached:8.1f}ms  ({tok_per_sec:5.1f} tok/s)")
        print(f"  Speedup:              {avg_naive/avg_cached:8.1f}x")
        print(f"  Avg exit depth:       {avg_depth:.1f} / {self.n_layers} layers")
        print(f"  Compute saved by EE:  {(1 - avg_depth/self.n_layers)*100:.0f}%")

        # Memory estimate
        model_mb = sum(p.nelement() * p.element_size() for p in self.model.parameters()) / 1e6
        cache_mb = sum(
            c.k.nelement() * c.k.element_size() + c.v.nelement() * c.v.element_size()
            for c in self.kv_caches
        ) / 1e6
        print(f"\n  Model memory:  {model_mb:.1f}MB")
        print(f"  KV cache:      {cache_mb:.1f}MB (pre-allocated for {self.config.max_seq_len} tokens)")
        print(f"  Total:         {model_mb + cache_mb:.1f}MB")


# ── OpenAI-compatible HTTP Server ───────────────────────────────────────────

def run_server(engine, host="0.0.0.0", port=8000):
    from http.server import HTTPServer, BaseHTTPRequestHandler
    from urllib.parse import urlparse
    import traceback

    MODEL_ID = "goby-25m"
    CREATED = int(time.time())

    def error_response(code, message, err_type="invalid_request_error"):
        return json.dumps({
            "error": {
                "message": message,
                "type": err_type,
                "param": None,
                "code": None,
            }
        }).encode()

    class Handler(BaseHTTPRequestHandler):

        def _cors(self):
            self.send_header("Access-Control-Allow-Origin", "*")
            self.send_header("Access-Control-Allow-Methods", "GET, POST, OPTIONS")
            self.send_header("Access-Control-Allow-Headers",
                             "Content-Type, Authorization, X-Request-ID")

        def _json_response(self, code, data):
            body = json.dumps(data).encode()
            self.send_response(code)
            self.send_header("Content-Type", "application/json")
            self.send_header("Content-Length", str(len(body)))
            self._cors()
            self.end_headers()
            self.wfile.write(body)

        def do_OPTIONS(self):
            self.send_response(204)
            self._cors()
            self.end_headers()

        def do_GET(self):
            path = urlparse(self.path).path

            if path == "/v1/models":
                self._json_response(200, {
                    "object": "list",
                    "data": [{
                        "id": MODEL_ID,
                        "object": "model",
                        "created": CREATED,
                        "owned_by": "goby",
                        "permission": [],
                        "root": MODEL_ID,
                        "parent": None,
                    }]
                })

            elif path.startswith("/v1/models/"):
                model_id = path.split("/v1/models/")[1]
                if model_id == MODEL_ID:
                    self._json_response(200, {
                        "id": MODEL_ID,
                        "object": "model",
                        "created": CREATED,
                        "owned_by": "goby",
                    })
                else:
                    self.send_response(404)
                    self.end_headers()
                    self.wfile.write(error_response(404, f"Model '{model_id}' not found", "not_found"))

            elif path == "/health" or path == "/":
                self._json_response(200, {"status": "ok", "model": MODEL_ID})

            else:
                self.send_response(404)
                self.end_headers()

        def do_POST(self):
            path = urlparse(self.path).path
            content_len = int(self.headers.get("Content-Length", 0))

            if path == "/v1/chat/completions":
                try:
                    body = json.loads(self.rfile.read(content_len))
                except json.JSONDecodeError:
                    self.send_response(400)
                    self.end_headers()
                    self.wfile.write(error_response(400, "Invalid JSON in request body"))
                    return

                messages = body.get("messages")
                if not messages:
                    self.send_response(400)
                    self.end_headers()
                    self.wfile.write(error_response(400, "'messages' is required"))
                    return

                # Stream requested — we don't support it, return error per OpenAI convention
                if body.get("stream", False):
                    self.send_response(400)
                    self.end_headers()
                    self.wfile.write(error_response(400, "Streaming is not supported by this model"))
                    return

                try:
                    result = engine.chat_completion(
                        messages=messages,
                        tools=body.get("tools"),
                        tool_choice=body.get("tool_choice", "auto"),
                        temperature=body.get("temperature", 0.7),
                        max_tokens=body.get("max_tokens", 256),
                        model=body.get("model"),
                        stop=body.get("stop"),
                    )
                    self._json_response(200, result)

                except Exception as e:
                    traceback.print_exc()
                    self.send_response(500)
                    self.end_headers()
                    self.wfile.write(error_response(500, str(e), "internal_error"))

            else:
                self.send_response(404)
                self.end_headers()

        def log_message(self, fmt, *args):
            print(f"[{self.log_date_time_string()}] {fmt % args}")

    srv = HTTPServer((host, port), Handler)
    print(f"\nGobyLLM OpenAI-compatible server")
    print(f"  http://{host}:{port}")
    print(f"  Model: {MODEL_ID} ({'INT8' if engine.quantized else 'FP32'})")
    print(f"  Early exit: {'ON' if engine.config.early_exit else 'OFF'}")
    print(f"\nEndpoints:")
    print(f"  POST /v1/chat/completions  — chat (tools, tool_choice supported)")
    print(f"  GET  /v1/models            — list models")
    print(f"  GET  /v1/models/{MODEL_ID} — model details")
    print(f"  GET  /health               — health check")
    print(f"\nUsage:")
    print(f'  from openai import OpenAI')
    print(f'  client = OpenAI(base_url="http://localhost:{port}/v1", api_key="not-needed")')
    srv.serve_forever()


# ── CLI ─────────────────────────────────────────────────────────────────────

def main():
    p = argparse.ArgumentParser(description="GobyLLM RPi Runner")
    p.add_argument("--checkpoint", default="checkpoints/best_model.pt")
    p.add_argument("--tokenizer", default="data/tokenizer.json")
    p.add_argument("--device", default="cpu")
    p.add_argument("--no-quantize", action="store_true", help="Disable INT8 quantization")
    p.add_argument("--serve", action="store_true")
    p.add_argument("--port", type=int, default=8000)
    p.add_argument("--benchmark", action="store_true")
    p.add_argument("--interactive", action="store_true")
    args = p.parse_args()

    engine = GobyRunner(
        args.checkpoint, args.tokenizer,
        quantize=not args.no_quantize,
        device=args.device,
    )

    if args.benchmark:
        engine.benchmark()
    elif args.serve:
        run_server(engine, port=args.port)
    elif args.interactive:
        tools = [
            {"type": "function", "function": {"name": "turn_on", "description": "Turn on a device",
             "parameters": {"type": "object", "properties": {"device": {"type": "string"}}, "required": ["device"]}}},
            {"type": "function", "function": {"name": "set_temperature", "description": "Set thermostat",
             "parameters": {"type": "object", "properties": {
                 "temperature": {"type": "number"}, "mode": {"type": "string", "enum": ["heat","cool","auto"]}
             }, "required": ["temperature", "mode"]}}},
        ]
        print(f"\nGobyLLM Interactive ({'INT8' if engine.quantized else 'FP32'}) — type 'quit' to exit")
        msgs = [{"role": "system", "content": "You are a helpful assistant."}]
        while True:
            inp = input("\nYou> ").strip()
            if inp.lower() in ("quit", "exit", "q"):
                break
            msgs.append({"role": "user", "content": inp})
            result = engine.chat_completion(msgs, tools=tools)
            ch = result["choices"][0]
            perf = result.get("_performance", {})
            if ch.get("_thinking"):
                print(f"  [think] {ch['_thinking'][:100]}")
            msg = ch["message"]
            if msg.get("tool_calls"):
                for tc in msg["tool_calls"]:
                    print(f"  [tool] {tc['function']['name']}({tc['function']['arguments']})")
            if msg.get("content"):
                print(f"Goby> {msg['content']}")
            if perf:
                print(f"  [{perf.get('tokens_per_sec',0)} tok/s, "
                      f"depth {perf.get('exit_depth','?')}/{perf.get('max_depth','?')}, "
                      f"{perf.get('compute_saved','?')} saved]")
            msgs.append(msg)
    else:
        print("Use --serve, --interactive, or --benchmark")


if __name__ == "__main__":
    main()

### `export_goby.py`

In [ ]:
%%writefile export_goby.py
"""Export GobyLLM checkpoint + tokenizer to a .goby binary for the C runtime."""

import argparse
import json
import os
import struct

import numpy as np
import torch

from config import GobyConfig
from model import GobyLLM


MAGIC   = 0x47425931  # "GBY1"
VERSION = 1

# ── Tokenizer helpers ───────────────────────────────────────────────────


def bytes_to_unicode():
    """GPT-2 / ByteLevel BPE byte-to-unicode mapping."""
    bs = list(range(ord("!"), ord("~") + 1))
    bs += list(range(ord("¡"), ord("¬") + 1))
    bs += list(range(ord("®"), ord("ÿ") + 1))
    cs = bs[:]
    n = 0
    for b in range(256):
        if b not in bs:
            bs.append(b)
            cs.append(256 + n)
            n += 1
    return {chr(c): b for b, c in zip(bs, cs)}


def export_tokenizer_data(tokenizer_path):
    """Extract vocab (as raw bytes per token) and merge pairs from tokenizer.json."""

    with open(tokenizer_path) as f:
        data = json.load(f)

    model_data = data["model"]
    vocab_str_to_id = model_data["vocab"]
    merges_raw = model_data.get("merges", [])

    # Build id → token string mapping
    id_to_str = {v: k for k, v in vocab_str_to_id.items()}
    vocab_size = max(id_to_str.keys()) + 1

    # Byte-level decoding: convert token unicode strings back to actual bytes
    unicode_to_byte = bytes_to_unicode()

    token_bytes = []
    for i in range(vocab_size):
        s = id_to_str.get(i, "")
        # Try to decode as byte-level
        try:
            raw = bytes([unicode_to_byte[c] for c in s])
        except (KeyError, ValueError):
            # Special token or unknown — store as UTF-8
            raw = s.encode("utf-8")
        token_bytes.append(raw)

    # Parse merges: each is either ["tokenA", "tokenB"] or "tokenA tokenB"
    merge_pairs = []
    for merge_entry in merges_raw:
        if isinstance(merge_entry, list):
            if len(merge_entry) == 2:
                a, b = merge_entry
            else:
                continue
        elif isinstance(merge_entry, str):
            parts = merge_entry.split(" ", 1)
            if len(parts) == 2:
                a, b = parts
            else:
                continue
        else:
            continue
        if a in vocab_str_to_id and b in vocab_str_to_id:
            merge_pairs.append((vocab_str_to_id[a], vocab_str_to_id[b]))

    # Build byte-to-initial-token lookup (single bytes)
    byte_to_token = [0] * 256
    byte_to_unicode_map = {v: k for k, v in bytes_to_unicode().items()}
    for byte_val in range(256):
        char = chr(byte_val)
        # Find the unicode char that represents this byte
        for uc, bv in unicode_to_byte.items():
            if bv == byte_val:
                if uc in vocab_str_to_id:
                    byte_to_token[byte_val] = vocab_str_to_id[uc]
                break

    return token_bytes, merge_pairs, byte_to_token, vocab_size


# ── Binary writer ───────────────────────────────────────────────────────


def write_goby(output_path, config, state_dict, rope_cos, rope_sin,
               token_bytes, merge_pairs, byte_to_token, vocab_size):
    """Write the .goby binary file."""
    with open(output_path, "wb") as f:
        # ── Header ──────────────────────────────────────────────────────
        f.write(struct.pack("I", MAGIC))
        f.write(struct.pack("I", VERSION))

        # Config
        f.write(struct.pack("i", config.vocab_size))
        f.write(struct.pack("i", config.d_model))
        f.write(struct.pack("i", config.n_layers))
        f.write(struct.pack("i", config.n_heads))
        f.write(struct.pack("i", config.n_kv_heads))
        f.write(struct.pack("i", config.ffn_hidden))
        f.write(struct.pack("i", config.max_seq_len))
        f.write(struct.pack("i", 1 if config.early_exit else 0))
        f.write(struct.pack("i", config.min_exit_layer))
        f.write(struct.pack("f", config.exit_threshold))

        # ── Tokenizer ──────────────────────────────────────────────────
        # Vocab
        f.write(struct.pack("I", vocab_size))
        for tb in token_bytes:
            f.write(struct.pack("H", len(tb)))
            f.write(tb)

        # Merges
        f.write(struct.pack("I", len(merge_pairs)))
        for a, b in merge_pairs:
            f.write(struct.pack("II", a, b))

        # Byte-to-token lookup
        for bt in byte_to_token:
            f.write(struct.pack("I", bt))

        # ── Model weights ─────────────────────────────────────────────
        def write_tensor(name):
            t = state_dict[name].float().cpu().numpy()
            f.write(t.tobytes())
            return t.size

        total_params = 0

        # Token embeddings (also used as lm_head)
        total_params += write_tensor("tok_emb.weight")

        # Transformer layers
        for i in range(config.n_layers):
            prefix = f"blocks.{i}."
            total_params += write_tensor(prefix + "norm.weight")
            total_params += write_tensor(prefix + "attn.wq.weight")
            total_params += write_tensor(prefix + "attn.wk.weight")
            total_params += write_tensor(prefix + "attn.wv.weight")
            total_params += write_tensor(prefix + "attn.wo.weight")
            total_params += write_tensor(prefix + "ffn.w_gate.weight")
            total_params += write_tensor(prefix + "ffn.w_up.weight")
            total_params += write_tensor(prefix + "ffn.w_down.weight")

            # Exit router
            if config.early_exit:
                total_params += write_tensor(f"exit_routers.{i}.weight")
                total_params += write_tensor(f"exit_routers.{i}.bias")

        # Final norm
        total_params += write_tensor("norm.weight")

        # RoPE buffers
        f.write(rope_cos.cpu().float().numpy().tobytes())
        f.write(rope_sin.cpu().float().numpy().tobytes())

    file_size = os.path.getsize(output_path)
    print(f"Exported {output_path}: {file_size / 1e6:.1f}MB ({total_params:,} params)")


# ── CLI ─────────────────────────────────────────────────────────────────


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--checkpoint", default="checkpoints/best_model.pt")
    parser.add_argument("--tokenizer", default="data/tokenizer.json")
    parser.add_argument("--output", default="goby.bin")
    args = parser.parse_args()

    # Load model
    ckpt = torch.load(args.checkpoint, map_location="cpu", weights_only=False)
    config = GobyConfig(**ckpt["config"])
    model = GobyLLM(config)
    model.load_state_dict(ckpt["model_state_dict"])
    model.eval()

    total, router = model.param_count()
    print(f"Model: {total:,} params ({total/1e6:.1f}M)")

    # Export tokenizer data
    token_bytes, merge_pairs, byte_to_token, vocab_size = export_tokenizer_data(args.tokenizer)
    print(f"Tokenizer: {vocab_size} tokens, {len(merge_pairs)} merges")

    # Write binary
    write_goby(
        args.output, config, model.state_dict(),
        model.rope_cos, model.rope_sin,
        token_bytes, merge_pairs, byte_to_token, vocab_size,
    )


if __name__ == "__main__":
    main()


### `goby.c`

In [ ]:
%%writefile goby.c
/*
 * goby.c — GobyLLM native inference engine
 *
 * Single-file C runtime for GobyLLM. No dependencies beyond libc and math.
 * Compiles on x86, ARM (RPi), Apple Silicon.
 *
 * Features:
 *   - mmap model loading (instant startup, OS manages memory paging)
 *   - KV cache (only processes new token each step)
 *   - Early exit (router-gated layer skipping)
 *   - NEON SIMD on ARM (auto-detected)
 *
 * Build:
 *   cc -O3 -o goby goby.c -lm                      # generic
 *   cc -O3 -march=native -o goby goby.c -lm         # native SIMD
 *   cc -O3 -mfpu=neon-fp-armv8 -o goby goby.c -lm   # RPi 4
 *
 * Usage:
 *   ./goby goby.bin -p "Turn on the lights"
 *   ./goby goby.bin -i                               # interactive
 */

#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <math.h>
#include <time.h>
#include <sys/mman.h>
#include <sys/stat.h>
#include <fcntl.h>
#include <unistd.h>

#ifdef __ARM_NEON
#include <arm_neon.h>
#endif

/* ═══════════════════════════════════════════════════════════════════════════
 *  DATA STRUCTURES
 * ═══════════════════════════════════════════════════════════════════════════ */

typedef struct {
    int vocab_size, d_model, n_layers, n_heads, n_kv_heads;
    int ffn_hidden, max_seq_len;
    int early_exit, min_exit_layer;
    float exit_threshold;
    /* derived */
    int head_dim, kv_dim, n_rep;
} Config;

typedef struct {
    float *norm_w;
    float *wq, *wk, *wv, *wo;
    float *ffn_gate, *ffn_up, *ffn_down;
    float *router_w;
    float *router_b;
} Layer;

typedef struct {
    Config cfg;
    float *tok_emb;      /* [vocab_size, d_model] — also used as lm_head */
    Layer *layers;
    float *final_norm_w;  /* [d_model] */
    float *rope_cos;      /* [max_seq_len, head_dim/2] */
    float *rope_sin;
} Model;

typedef struct {
    char **tokens;        /* id → byte string */
    int *token_lens;      /* id → length */
    int vocab_size;
    int *merges;          /* [n_merges * 2] pairs of token IDs */
    int n_merges;
    int byte_to_token[256];
} Tokenizer;

typedef struct {
    float *x, *xb, *xb2;        /* [d_model] buffers */
    float *q;                     /* [n_heads * head_dim] */
    float *att;                   /* [n_heads * max_seq_len] */
    float *ffn_buf1, *ffn_buf2;  /* [ffn_hidden] */
    float *logits;                /* [vocab_size] */
    float *key_cache;             /* [n_layers * max_seq_len * kv_dim] */
    float *val_cache;
} RunState;

/* ═══════════════════════════════════════════════════════════════════════════
 *  MATH OPS
 * ═══════════════════════════════════════════════════════════════════════════ */

static void rmsnorm(float *out, const float *x, const float *w, int size) {
    float ss = 0.0f;
    for (int i = 0; i < size; i++) ss += x[i] * x[i];
    ss = 1.0f / sqrtf(ss / size + 1e-5f);
    for (int i = 0; i < size; i++) out[i] = x[i] * ss * w[i];
}

static void matmul(float *out, const float *x, const float *w, int n, int d) {
    /* out[i] = dot(w[i], x) for i in 0..n-1.  w is [n, d], x is [d]. */
    #pragma omp parallel for
    for (int i = 0; i < n; i++) {
        float sum = 0.0f;
        const float *wi = w + i * d;
#ifdef __ARM_NEON
        int j = 0;
        float32x4_t sv = vdupq_n_f32(0.0f);
        for (; j + 4 <= d; j += 4) {
            sv = vmlaq_f32(sv, vld1q_f32(x + j), vld1q_f32(wi + j));
        }
        sum = vaddvq_f32(sv);
        for (; j < d; j++) sum += x[j] * wi[j];
#else
        for (int j = 0; j < d; j++) sum += x[j] * wi[j];
#endif
        out[i] = sum;
    }
}

static float silu_f(float x) { return x / (1.0f + expf(-x)); }

static void softmax(float *x, int size) {
    float max_val = x[0];
    for (int i = 1; i < size; i++) if (x[i] > max_val) max_val = x[i];
    float sum = 0.0f;
    for (int i = 0; i < size; i++) { x[i] = expf(x[i] - max_val); sum += x[i]; }
    for (int i = 0; i < size; i++) x[i] /= sum;
}

/* ═══════════════════════════════════════════════════════════════════════════
 *  MODEL LOADING (mmap)
 * ═══════════════════════════════════════════════════════════════════════════ */

static void *file_data;
static size_t file_size;

static void *read_ptr(void **cursor, size_t bytes) {
    void *p = *cursor;
    *cursor = (char *)*cursor + bytes;
    return p;
}

static int read_int(void **c) { return *(int *)read_ptr(c, 4); }
static unsigned read_uint(void **c) { return *(unsigned *)read_ptr(c, 4); }
static float read_float(void **c) { return *(float *)read_ptr(c, 4); }
static unsigned short read_ushort(void **c) { return *(unsigned short *)read_ptr(c, 2); }

static float *read_floats(void **c, int count) {
    return (float *)read_ptr(c, count * sizeof(float));
}

static int load_model(const char *path, Model *m, Tokenizer *tok) {
    int fd = open(path, O_RDONLY);
    if (fd < 0) { perror("open"); return -1; }
    struct stat st;
    fstat(fd, &st);
    file_size = st.st_size;
    file_data = mmap(NULL, file_size, PROT_READ, MAP_PRIVATE, fd, 0);
    close(fd);
    if (file_data == MAP_FAILED) { perror("mmap"); return -1; }

    void *cursor = file_data;

    /* Header */
    unsigned magic = read_uint(&cursor);
    if (magic != 0x47425931) { fprintf(stderr, "bad magic\n"); return -1; }
    read_uint(&cursor); /* version */

    /* Config */
    Config *c = &m->cfg;
    c->vocab_size   = read_int(&cursor);
    c->d_model      = read_int(&cursor);
    c->n_layers     = read_int(&cursor);
    c->n_heads      = read_int(&cursor);
    c->n_kv_heads   = read_int(&cursor);
    c->ffn_hidden   = read_int(&cursor);
    c->max_seq_len  = read_int(&cursor);
    c->early_exit   = read_int(&cursor);
    c->min_exit_layer = read_int(&cursor);
    c->exit_threshold = read_float(&cursor);
    c->head_dim = c->d_model / c->n_heads;
    c->kv_dim = c->n_kv_heads * c->head_dim;
    c->n_rep = c->n_heads / c->n_kv_heads;

    /* Tokenizer — vocab */
    tok->vocab_size = read_uint(&cursor);
    tok->tokens = (char **)malloc(tok->vocab_size * sizeof(char *));
    tok->token_lens = (int *)malloc(tok->vocab_size * sizeof(int));
    for (int i = 0; i < tok->vocab_size; i++) {
        int len = read_ushort(&cursor);
        tok->token_lens[i] = len;
        tok->tokens[i] = (char *)malloc(len + 1);
        memcpy(tok->tokens[i], read_ptr(&cursor, len), len);
        tok->tokens[i][len] = '\0';
    }

    /* Tokenizer — merges */
    tok->n_merges = read_uint(&cursor);
    tok->merges = (int *)malloc(tok->n_merges * 2 * sizeof(int));
    for (int i = 0; i < tok->n_merges; i++) {
        tok->merges[i * 2]     = read_uint(&cursor);
        tok->merges[i * 2 + 1] = read_uint(&cursor);
    }

    /* Tokenizer — byte-to-token */
    for (int i = 0; i < 256; i++) tok->byte_to_token[i] = read_uint(&cursor);

    /* Weights — tok_emb */
    m->tok_emb = read_floats(&cursor, c->vocab_size * c->d_model);

    /* Weights — layers */
    m->layers = (Layer *)malloc(c->n_layers * sizeof(Layer));
    for (int i = 0; i < c->n_layers; i++) {
        Layer *l = &m->layers[i];
        l->norm_w   = read_floats(&cursor, c->d_model);
        l->wq       = read_floats(&cursor, c->n_heads * c->head_dim * c->d_model);
        l->wk       = read_floats(&cursor, c->kv_dim * c->d_model);
        l->wv       = read_floats(&cursor, c->kv_dim * c->d_model);
        l->wo       = read_floats(&cursor, c->d_model * c->d_model);
        l->ffn_gate = read_floats(&cursor, c->ffn_hidden * c->d_model);
        l->ffn_up   = read_floats(&cursor, c->ffn_hidden * c->d_model);
        l->ffn_down = read_floats(&cursor, c->d_model * c->ffn_hidden);
        if (c->early_exit) {
            l->router_w = read_floats(&cursor, c->d_model);  /* [1, d_model] */
            l->router_b = read_floats(&cursor, 1);
        }
    }

    m->final_norm_w = read_floats(&cursor, c->d_model);
    m->rope_cos = read_floats(&cursor, c->max_seq_len * c->head_dim / 2);
    m->rope_sin = read_floats(&cursor, c->max_seq_len * c->head_dim / 2);

    return 0;
}

/* ═══════════════════════════════════════════════════════════════════════════
 *  RUN STATE
 * ═══════════════════════════════════════════════════════════════════════════ */

static void init_run_state(RunState *s, Config *c) {
    int d = c->d_model;
    s->x        = (float *)calloc(d, sizeof(float));
    s->xb       = (float *)calloc(d, sizeof(float));
    s->xb2      = (float *)calloc(d, sizeof(float));
    s->q        = (float *)calloc(c->n_heads * c->head_dim, sizeof(float));
    s->att      = (float *)calloc(c->n_heads * c->max_seq_len, sizeof(float));
    s->ffn_buf1 = (float *)calloc(c->ffn_hidden, sizeof(float));
    s->ffn_buf2 = (float *)calloc(c->ffn_hidden, sizeof(float));
    s->logits   = (float *)calloc(c->vocab_size, sizeof(float));
    s->key_cache = (float *)calloc(c->n_layers * c->max_seq_len * c->kv_dim, sizeof(float));
    s->val_cache = (float *)calloc(c->n_layers * c->max_seq_len * c->kv_dim, sizeof(float));
    /* exit depth determined at runtime */
}

static void reset_state(RunState *s, Config *c) {
    memset(s->key_cache, 0, c->n_layers * c->max_seq_len * c->kv_dim * sizeof(float));
    memset(s->val_cache, 0, c->n_layers * c->max_seq_len * c->kv_dim * sizeof(float));
    /* exit depth determined at runtime */
}

/* ═══════════════════════════════════════════════════════════════════════════
 *  FORWARD PASS (single token, with KV cache)
 * ═══════════════════════════════════════════════════════════════════════════ */

static void forward(Model *m, RunState *s, int token, int pos, int max_layer) {
    Config *c = &m->cfg;
    int d = c->d_model;
    int hd = c->head_dim;
    int half_hd = hd / 2;
    int kv_dim = c->kv_dim;

    /* Token embedding */
    memcpy(s->x, m->tok_emb + token * d, d * sizeof(float));

    for (int layer = 0; layer < max_layer; layer++) {
        Layer *l = &m->layers[layer];

        /* RMSNorm */
        rmsnorm(s->xb, s->x, l->norm_w, d);

        /* ── Attention ──────────────────────────────────────────────── */
        /* Q, K, V projections */
        matmul(s->q, s->xb, l->wq, c->n_heads * hd, d);
        float *kv_k = s->key_cache + layer * c->max_seq_len * kv_dim + pos * kv_dim;
        float *kv_v = s->val_cache + layer * c->max_seq_len * kv_dim + pos * kv_dim;
        matmul(kv_k, s->xb, l->wk, kv_dim, d);
        matmul(kv_v, s->xb, l->wv, kv_dim, d);

        /* RoPE */
        float *cos_row = m->rope_cos + pos * half_hd;
        float *sin_row = m->rope_sin + pos * half_hd;
        for (int h = 0; h < c->n_heads; h++) {
            float *qh = s->q + h * hd;
            for (int j = 0; j < half_hd; j++) {
                float q0 = qh[j], q1 = qh[j + half_hd];
                qh[j]           = q0 * cos_row[j] - q1 * sin_row[j];
                qh[j + half_hd] = q1 * cos_row[j] + q0 * sin_row[j];
            }
        }
        for (int h = 0; h < c->n_kv_heads; h++) {
            float *kh = kv_k + h * hd;
            for (int j = 0; j < half_hd; j++) {
                float k0 = kh[j], k1 = kh[j + half_hd];
                kh[j]           = k0 * cos_row[j] - k1 * sin_row[j];
                kh[j + half_hd] = k1 * cos_row[j] + k0 * sin_row[j];
            }
        }

        /* Grouped query attention with KV cache */
        memset(s->xb2, 0, d * sizeof(float));
        for (int h = 0; h < c->n_heads; h++) {
            int kv_h = h / c->n_rep;  /* which KV head this Q head uses */
            float *qh = s->q + h * hd;
            float *att_h = s->att + h * c->max_seq_len;

            /* Compute attention scores for all cached positions */
            for (int t = 0; t <= pos; t++) {
                float *kt = s->key_cache + layer * c->max_seq_len * kv_dim + t * kv_dim + kv_h * hd;
                float score = 0.0f;
                for (int j = 0; j < hd; j++) score += qh[j] * kt[j];
                att_h[t] = score / sqrtf((float)hd);
            }

            /* Softmax over [0..pos] */
            softmax(att_h, pos + 1);

            /* Weighted sum of values */
            float *oh = s->xb2 + h * hd;
            for (int t = 0; t <= pos; t++) {
                float *vt = s->val_cache + layer * c->max_seq_len * kv_dim + t * kv_dim + kv_h * hd;
                float a = att_h[t];
                for (int j = 0; j < hd; j++) oh[j] += a * vt[j];
            }
        }

        /* Output projection (attn) */
        float attn_out[d];
        matmul(attn_out, s->xb2, l->wo, d, d);

        /* ── FFN (SwiGLU, parallel residual) ────────────────────────── */
        matmul(s->ffn_buf1, s->xb, l->ffn_gate, c->ffn_hidden, d);
        matmul(s->ffn_buf2, s->xb, l->ffn_up, c->ffn_hidden, d);
        for (int j = 0; j < c->ffn_hidden; j++)
            s->ffn_buf1[j] = silu_f(s->ffn_buf1[j]) * s->ffn_buf2[j];
        float ffn_out[d];
        matmul(ffn_out, s->ffn_buf1, l->ffn_down, d, c->ffn_hidden);

        /* Parallel residual: x = x + attn_out + ffn_out */
        for (int j = 0; j < d; j++)
            s->x[j] += attn_out[j] + ffn_out[j];
    }

    /* Final norm + lm_head (tok_emb weights, transposed) */
    rmsnorm(s->xb, s->x, m->final_norm_w, d);
    matmul(s->logits, s->xb, m->tok_emb, c->vocab_size, d);
}

/* Early exit: check router confidence after full prompt processing */
static int check_exit_depth(Model *m, RunState *s) {
    Config *c = &m->cfg;
    if (!c->early_exit) return c->n_layers;
    for (int i = c->min_exit_layer; i < c->n_layers - 1; i++) {
        Layer *l = &m->layers[i];
        float dot = l->router_b[0];
        for (int j = 0; j < c->d_model; j++) dot += s->x[j] * l->router_w[j];
        float conf = 1.0f / (1.0f + expf(-dot));
        if (conf > c->exit_threshold) return i + 1;
    }
    return c->n_layers;
}

/* ═══════════════════════════════════════════════════════════════════════════
 *  TOKENIZER
 * ═══════════════════════════════════════════════════════════════════════════ */

static int *bpe_encode(Tokenizer *tok, const char *text, int *n_tokens) {
    int len = strlen(text);
    int *tokens = (int *)malloc((len + 128) * sizeof(int));
    int n = 0;

    /* Check for special tokens at the beginning */
    const char *specials[] = {
        "<|im_start|>", "<|im_end|>", "<think>", "</think>",
        "<tool_call>", "</tool_call>", "<pad>", NULL
    };
    int special_ids[] = {1, 2, 3, 4, 5, 6, 0};

    int i = 0;
    while (i < len) {
        /* Try special tokens */
        int matched = 0;
        for (int s = 0; specials[s]; s++) {
            int slen = strlen(specials[s]);
            if (i + slen <= len && strncmp(text + i, specials[s], slen) == 0) {
                tokens[n++] = special_ids[s];
                i += slen;
                matched = 1;
                break;
            }
        }
        if (matched) continue;

        /* Map byte to initial token */
        tokens[n++] = tok->byte_to_token[(unsigned char)text[i]];
        i++;
    }

    /* Apply BPE merges */
    for (int m = 0; m < tok->n_merges; m++) {
        int id1 = tok->merges[m * 2];
        int id2 = tok->merges[m * 2 + 1];
        /* Find merged token ID: the token whose bytes = bytes(id1) + bytes(id2) */
        /* For simplicity, search vocab. In production, pre-build a merge→result map. */
        int result_id = -1;
        int len1 = tok->token_lens[id1];
        int len2 = tok->token_lens[id2];
        int merged_len = len1 + len2;
        for (int v = 0; v < tok->vocab_size; v++) {
            if (tok->token_lens[v] == merged_len &&
                memcmp(tok->tokens[v], tok->tokens[id1], len1) == 0 &&
                memcmp(tok->tokens[v] + len1, tok->tokens[id2], len2) == 0) {
                result_id = v;
                break;
            }
        }
        if (result_id < 0) continue;

        for (int j = 0; j < n - 1; j++) {
            if (tokens[j] == id1 && tokens[j + 1] == id2) {
                tokens[j] = result_id;
                memmove(&tokens[j + 1], &tokens[j + 2], (n - j - 2) * sizeof(int));
                n--;
                j--;
            }
        }
    }

    *n_tokens = n;
    return tokens;
}

static void decode_token(Tokenizer *tok, int id, char *buf, int buf_size) {
    if (id >= 0 && id < tok->vocab_size) {
        int len = tok->token_lens[id];
        if (len >= buf_size) len = buf_size - 1;
        memcpy(buf, tok->tokens[id], len);
        buf[len] = '\0';
    } else {
        buf[0] = '\0';
    }
}

/* ═══════════════════════════════════════════════════════════════════════════
 *  SAMPLING
 * ═══════════════════════════════════════════════════════════════════════════ */

static int sample_topk(float *logits, int vocab_size, float temperature, int top_k) {
    if (temperature < 1e-6f) {
        /* Greedy */
        int best = 0;
        for (int i = 1; i < vocab_size; i++)
            if (logits[i] > logits[best]) best = i;
        return best;
    }

    /* Apply temperature */
    for (int i = 0; i < vocab_size; i++) logits[i] /= temperature;

    /* Top-K: find Kth largest, mask below it */
    if (top_k > 0 && top_k < vocab_size) {
        float threshold = -1e9f;
        /* Partial sort: find top_k-th value */
        float *tmp = (float *)malloc(vocab_size * sizeof(float));
        memcpy(tmp, logits, vocab_size * sizeof(float));
        /* Simple selection for small K */
        for (int i = 0; i < top_k; i++) {
            int best = i;
            for (int j = i + 1; j < vocab_size; j++)
                if (tmp[j] > tmp[best]) best = j;
            float t = tmp[i]; tmp[i] = tmp[best]; tmp[best] = t;
        }
        threshold = tmp[top_k - 1];
        free(tmp);
        for (int i = 0; i < vocab_size; i++)
            if (logits[i] < threshold) logits[i] = -1e9f;
    }

    softmax(logits, vocab_size);

    /* Sample from distribution */
    float r = (float)rand() / (float)RAND_MAX;
    float cumsum = 0.0f;
    for (int i = 0; i < vocab_size; i++) {
        cumsum += logits[i];
        if (cumsum >= r) return i;
    }
    return vocab_size - 1;
}

/* ═══════════════════════════════════════════════════════════════════════════
 *  GENERATION
 * ═══════════════════════════════════════════════════════════════════════════ */

typedef struct {
    int exit_depth;
    int tokens_generated;
    double prompt_ms;
    double gen_ms;
    double tokens_per_sec;
} GenStats;

static void generate(Model *m, Tokenizer *tok, RunState *s,
                     int *prompt, int prompt_len, int max_tokens,
                     float temperature, int top_k, GenStats *stats) {
    Config *c = &m->cfg;
    reset_state(s, c);

    /* Phase 1: Process prompt tokens (all layers, fills KV cache) */
    struct timespec t0, t1;
    clock_gettime(CLOCK_MONOTONIC, &t0);

    for (int i = 0; i < prompt_len; i++) {
        forward(m, s, prompt[i], i, c->n_layers);
    }

    /* Determine early exit depth from router on last prompt token */
    int exit_depth = check_exit_depth(m, s);

    clock_gettime(CLOCK_MONOTONIC, &t1);
    double prompt_ms = (t1.tv_sec - t0.tv_sec) * 1000.0 + (t1.tv_nsec - t0.tv_nsec) / 1e6;

    /* Phase 2: Generate tokens with KV cache at fixed depth */
    clock_gettime(CLOCK_MONOTONIC, &t0);

    int pos = prompt_len;
    int next_token = sample_topk(s->logits, c->vocab_size, temperature, top_k);
    int gen_count = 0;
    char decoded[256];

    for (int t = 0; t < max_tokens; t++) {
        if (next_token == 2) break;  /* <|im_end|> */

        decode_token(tok, next_token, decoded, sizeof(decoded));
        printf("%s", decoded);
        fflush(stdout);
        gen_count++;

        if (pos >= c->max_seq_len - 1) break;

        forward(m, s, next_token, pos, exit_depth);
        next_token = sample_topk(s->logits, c->vocab_size, temperature, top_k);
        pos++;
    }
    printf("\n");

    clock_gettime(CLOCK_MONOTONIC, &t1);
    double gen_ms = (t1.tv_sec - t0.tv_sec) * 1000.0 + (t1.tv_nsec - t0.tv_nsec) / 1e6;

    if (stats) {
        stats->exit_depth = exit_depth;
        stats->tokens_generated = gen_count;
        stats->prompt_ms = prompt_ms;
        stats->gen_ms = gen_ms;
        stats->tokens_per_sec = gen_count > 0 ? gen_count / (gen_ms / 1000.0) : 0;
    }
}

/* ═══════════════════════════════════════════════════════════════════════════
 *  BENCHMARK
 * ═══════════════════════════════════════════════════════════════════════════ */

static void benchmark(Model *m, Tokenizer *tok, RunState *s) {
    Config *c = &m->cfg;
    int prompt_len = 32, gen_len = 64, n_runs = 5;

    printf("\n══════════════════════════════════════════\n");
    printf("GobyLLM Benchmark (C runtime)\n");
    printf("  Model: %dM params, %d layers\n", (int)(c->vocab_size * c->d_model / 1e6 + c->n_layers * 2.0), c->n_layers);
    printf("  Prompt: %d tokens, Generate: %d tokens\n", prompt_len, gen_len);
    printf("══════════════════════════════════════════\n");

    int prompt[32];
    for (int i = 0; i < prompt_len; i++) prompt[i] = (rand() % (c->vocab_size - 7)) + 7;

    double total_prompt = 0, total_gen = 0;
    int total_depth = 0;

    for (int r = 0; r < n_runs; r++) {
        GenStats stats = {0};
        /* Suppress output during benchmark */
        FILE *devnull = fopen("/dev/null", "w");
        FILE *saved_stdout = stdout;
        stdout = devnull;
        generate(m, tok, s, prompt, prompt_len, gen_len, 0.8f, 40, &stats);
        stdout = saved_stdout;
        fclose(devnull);

        total_prompt += stats.prompt_ms;
        total_gen += stats.gen_ms;
        total_depth += stats.exit_depth;
    }

    double avg_prompt = total_prompt / n_runs;
    double avg_gen = total_gen / n_runs;
    double avg_depth = (double)total_depth / n_runs;
    double tok_per_sec = gen_len / (avg_gen / 1000.0);

    printf("\n  Prompt (%d tokens):   %7.1f ms\n", prompt_len, avg_prompt);
    printf("  Generate (%d tokens): %7.1f ms  (%.1f tok/s)\n", gen_len, avg_gen, tok_per_sec);
    printf("  Avg exit depth:      %.1f / %d layers\n", avg_depth, c->n_layers);
    printf("  Compute saved (EE):  %.0f%%\n", (1.0 - avg_depth / c->n_layers) * 100);

    size_t model_bytes = file_size;
    size_t cache_bytes = c->n_layers * c->max_seq_len * c->kv_dim * 2 * sizeof(float);
    printf("\n  Model file:  %.1f MB\n", model_bytes / 1e6);
    printf("  KV cache:    %.1f MB\n", cache_bytes / 1e6);
    printf("  Total RAM:   %.1f MB\n", (model_bytes + cache_bytes) / 1e6);
}

/* ═══════════════════════════════════════════════════════════════════════════
 *  MAIN
 * ═══════════════════════════════════════════════════════════════════════════ */

int main(int argc, char **argv) {
    if (argc < 2) {
        fprintf(stderr, "GobyLLM C Runtime\n\n");
        fprintf(stderr, "Usage:\n");
        fprintf(stderr, "  %s model.bin -p \"prompt\"      Generate from prompt\n", argv[0]);
        fprintf(stderr, "  %s model.bin -i               Interactive mode\n", argv[0]);
        fprintf(stderr, "  %s model.bin -b               Benchmark\n", argv[0]);
        fprintf(stderr, "\nOptions:\n");
        fprintf(stderr, "  -t <float>   Temperature (default 0.7)\n");
        fprintf(stderr, "  -k <int>     Top-K (default 40)\n");
        fprintf(stderr, "  -n <int>     Max tokens (default 256)\n");
        return 1;
    }

    srand(time(NULL));

    const char *model_path = argv[1];
    const char *prompt = NULL;
    int interactive = 0, do_bench = 0;
    float temperature = 0.7f;
    int top_k = 40, max_tokens = 256;

    for (int i = 2; i < argc; i++) {
        if (strcmp(argv[i], "-p") == 0 && i + 1 < argc) prompt = argv[++i];
        else if (strcmp(argv[i], "-i") == 0) interactive = 1;
        else if (strcmp(argv[i], "-b") == 0) do_bench = 1;
        else if (strcmp(argv[i], "-t") == 0 && i + 1 < argc) temperature = atof(argv[++i]);
        else if (strcmp(argv[i], "-k") == 0 && i + 1 < argc) top_k = atoi(argv[++i]);
        else if (strcmp(argv[i], "-n") == 0 && i + 1 < argc) max_tokens = atoi(argv[++i]);
    }

    Model model;
    Tokenizer tok;
    printf("Loading %s...\n", model_path);
    if (load_model(model_path, &model, &tok) != 0) return 1;

    Config *c = &model.cfg;
    printf("GobyLLM: vocab=%d, d=%d, layers=%d, heads=%dQ/%dKV, ffn=%d\n",
           c->vocab_size, c->d_model, c->n_layers, c->n_heads, c->n_kv_heads, c->ffn_hidden);
    printf("Early exit: %s (min_layer=%d, threshold=%.2f)\n",
           c->early_exit ? "ON" : "OFF", c->min_exit_layer, c->exit_threshold);

    RunState state;
    init_run_state(&state, c);

    if (do_bench) {
        benchmark(&model, &tok, &state);
    } else if (interactive) {
        char input[4096];
        printf("\nGobyLLM Interactive (type 'quit' to exit)\n");
        while (1) {
            printf("\nYou> ");
            if (!fgets(input, sizeof(input), stdin)) break;
            input[strcspn(input, "\n")] = '\0';
            if (strcmp(input, "quit") == 0 || strcmp(input, "exit") == 0) break;
            if (input[0] == '\0') continue;

            /* Build chat prompt */
            char full_prompt[8192];
            snprintf(full_prompt, sizeof(full_prompt),
                     "<|im_start|>system\nYou are a helpful assistant.<|im_end|>\n"
                     "<|im_start|>user\n%s<|im_end|>\n"
                     "<|im_start|>assistant\n", input);

            int n_tokens;
            int *tokens = bpe_encode(&tok, full_prompt, &n_tokens);

            GenStats stats;
            printf("Goby> ");
            generate(&model, &tok, &state, tokens, n_tokens, max_tokens,
                     temperature, top_k, &stats);
            printf("  [%.1f tok/s, depth %d/%d, %.0f%% saved]\n",
                   stats.tokens_per_sec, stats.exit_depth, c->n_layers,
                   (1.0 - (double)stats.exit_depth / c->n_layers) * 100);
            free(tokens);
        }
    } else if (prompt) {
        /* Build chat prompt */
        char full_prompt[8192];
        snprintf(full_prompt, sizeof(full_prompt),
                 "<|im_start|>system\nYou are a helpful assistant.<|im_end|>\n"
                 "<|im_start|>user\n%s<|im_end|>\n"
                 "<|im_start|>assistant\n", prompt);

        int n_tokens;
        int *tokens = bpe_encode(&tok, full_prompt, &n_tokens);

        GenStats stats;
        generate(&model, &tok, &state, tokens, n_tokens, max_tokens,
                 temperature, top_k, &stats);
        fprintf(stderr, "\n[%d tokens, %.1f tok/s, depth %d/%d, prompt %.0fms, gen %.0fms]\n",
                stats.tokens_generated, stats.tokens_per_sec,
                stats.exit_depth, c->n_layers, stats.prompt_ms, stats.gen_ms);
        free(tokens);
    }

    /* Cleanup */
    munmap(file_data, file_size);
    return 0;
}


### `Makefile`

In [ ]:
%%writefile Makefile
CC = cc
CFLAGS = -O3 -march=native -Wall
LDFLAGS = -lm

# Auto-detect ARM NEON
UNAME_M := $(shell uname -m)
ifeq ($(UNAME_M),aarch64)
    CFLAGS += -mfpu=neon-fp-armv8
endif
ifeq ($(UNAME_M),armv7l)
    CFLAGS += -mfpu=neon-vfpv4
endif

.PHONY: all clean

all: goby

goby: goby.c
	$(CC) $(CFLAGS) -o $@ $< $(LDFLAGS)
	@echo "Built: ./goby"
	@echo "Usage: ./goby goby.bin -p 'Hello' | -i | -b"

# Debug build (no optimization, address sanitizer)
debug: goby.c
	$(CC) -g -O0 -fsanitize=address -o goby_debug $< $(LDFLAGS)

clean:
	rm -f goby goby_debug


## 3. Prepare Data

Downloads Stanford Alpaca (52K samples), generates tool-calling data (15K samples),
trains a BPE tokenizer, and merges everything.

In [ ]:
!python prepare_data.py

In [ ]:
# Preview a sample
import json
with open('data/train.jsonl') as f:
    s = json.loads(f.readline())
print(f'Source: {s["source"]}')
print(s['text'][:500])
print('...')

## 4. Verify Architecture

In [ ]:
from config import GobyConfig
from model import GobyLLM
import torch

config = GobyConfig()
model = GobyLLM(config)
print(model.param_summary())
print(f'  GQA: {config.n_heads}Q / {config.n_kv_heads}KV')
print(f'  Parallel residual: {config.parallel_residual}')
print(f'  Early exit: min_layer={config.min_exit_layer}, threshold={config.exit_threshold}')

x = torch.randint(0, config.vocab_size, (2, 64))
logits, _ = model(x)
print(f'  Forward: {x.shape} -> {logits.shape}')
del model

## 5. Train

10,000 steps. Takes ~15-20 min on T4.

In [ ]:
from train import train
train()

## 6. Test Tool Calling

Define a rich set of tools and test diverse calling patterns:
direct commands, implicit intent, multi-tool selection, refusal, parameter precision.

In [ ]:
from inference import GobyInference
import json, torch

engine = GobyInference(
    'checkpoints/best_model.pt', 'data/tokenizer.json',
    device='cuda' if torch.cuda.is_available() else 'cpu'
)

# ── Tool definitions ────────────────────────────────────────────
TOOLS = [
    {'type': 'function', 'function': {
        'name': 'turn_on',
        'description': 'Turn on a device (light, fan, appliance, etc.)',
        'parameters': {'type': 'object', 'properties': {
            'device': {'type': 'string', 'description': 'Device name, e.g. kitchen lights'},
        }, 'required': ['device']},
    }},
    {'type': 'function', 'function': {
        'name': 'turn_off',
        'description': 'Turn off a device',
        'parameters': {'type': 'object', 'properties': {
            'device': {'type': 'string', 'description': 'Device name'},
        }, 'required': ['device']},
    }},
    {'type': 'function', 'function': {
        'name': 'set_temperature',
        'description': 'Set thermostat target temperature and HVAC mode',
        'parameters': {'type': 'object', 'properties': {
            'temperature': {'type': 'number', 'description': 'Target temp in celsius'},
            'mode': {'type': 'string', 'enum': ['heat', 'cool', 'auto'], 'description': 'HVAC mode'},
        }, 'required': ['temperature', 'mode']},
    }},
    {'type': 'function', 'function': {
        'name': 'set_brightness',
        'description': 'Set light brightness level 0-100',
        'parameters': {'type': 'object', 'properties': {
            'device': {'type': 'string', 'description': 'Light device name'},
            'level': {'type': 'integer', 'description': 'Brightness 0-100'},
        }, 'required': ['device', 'level']},
    }},
    {'type': 'function', 'function': {
        'name': 'send_alert',
        'description': 'Send alert to monitoring dashboard',
        'parameters': {'type': 'object', 'properties': {
            'level': {'type': 'string', 'enum': ['info', 'warning', 'critical', 'emergency']},
            'message': {'type': 'string', 'description': 'Alert message'},
            'zone': {'type': 'string', 'description': 'Zone or location'},
        }, 'required': ['level', 'message', 'zone']},
    }},
    {'type': 'function', 'function': {
        'name': 'lock_door',
        'description': 'Lock a door',
        'parameters': {'type': 'object', 'properties': {
            'door': {'type': 'string', 'description': 'Door name'},
        }, 'required': ['door']},
    }},
    {'type': 'function', 'function': {
        'name': 'set_timer',
        'description': 'Set a countdown timer',
        'parameters': {'type': 'object', 'properties': {
            'duration': {'type': 'string', 'description': 'Timer duration, e.g. 10 minutes'},
            'label': {'type': 'string', 'description': 'What the timer is for'},
        }, 'required': ['duration', 'label']},
    }},
    {'type': 'function', 'function': {
        'name': 'start_pump',
        'description': 'Start a pump at specified flow rate',
        'parameters': {'type': 'object', 'properties': {
            'pump_id': {'type': 'string'},
            'flow_rate': {'type': 'string', 'enum': ['low', 'medium', 'high', 'max']},
        }, 'required': ['pump_id', 'flow_rate']},
    }},
]

SYSTEM = 'You are a helpful assistant. Use the provided tools when appropriate.'

def test(prompt, expect_tool=None, expect_param=None, desc=''):
    """Run a test case and report results."""
    r = engine.chat_completion(
        [{'role': 'system', 'content': SYSTEM},
         {'role': 'user', 'content': prompt}],
        tools=TOOLS
    )
    ch = r['choices'][0]
    msg = ch['message']
    tc = msg.get('tool_calls', [None])[0] if msg.get('tool_calls') else None
    tool_name = tc['function']['name'] if tc else None
    tool_args = json.loads(tc['function']['arguments']) if tc else {}

    # Check expectations
    status = ''
    if expect_tool is not None:
        if expect_tool is False:
            status = '✓ PASS' if tool_name is None else f'✗ FAIL (got {tool_name}, expected no tool)'
        else:
            if tool_name == expect_tool:
                status = '✓ PASS'
                if expect_param:
                    for k, v in expect_param.items():
                        if str(tool_args.get(k, '')).lower() != str(v).lower():
                            status = f'✗ PARAM ({k}: got {tool_args.get(k)}, expected {v})'
            else:
                status = f'✗ FAIL (got {tool_name}, expected {expect_tool})'

    print(f'{status:20s} | {desc}')
    print(f'  Prompt:  {prompt}')
    if tc:
        print(f'  Tool:    {tool_name}({json.dumps(tool_args)})')
    if msg.get('content'):
        print(f'  Reply:   {msg["content"][:120]}')
    if ch.get('_thinking'):
        print(f'  Think:   {ch["_thinking"][:120]}')
    print()
    return tool_name, tool_args

### Direct commands — explicit tool invocation

In [ ]:
test('Turn on the kitchen lights',
     expect_tool='turn_on', expect_param={'device': 'kitchen lights'},
     desc='Direct: turn on specific device')

test('Turn off the bedroom fan',
     expect_tool='turn_off', expect_param={'device': 'bedroom fan'},
     desc='Direct: turn off specific device')

test('Lock the front door',
     expect_tool='lock_door', expect_param={'door': 'front door'},
     desc='Direct: lock specific door')

test('Set the temperature to 22 degrees, cooling mode',
     expect_tool='set_temperature', expect_param={'temperature': 22, 'mode': 'cool'},
     desc='Direct: thermostat with exact values')

test('Dim the living room lights to 30%',
     expect_tool='set_brightness', expect_param={'level': 30},
     desc='Direct: brightness with exact level')

test('Set a timer for 15 minutes for the pasta',
     expect_tool='set_timer', expect_param={'duration': '15 minutes'},
     desc='Direct: timer with duration and label')

test('Start pump_3 at high flow',
     expect_tool='start_pump', expect_param={'pump_id': 'pump_3', 'flow_rate': 'high'},
     desc='Direct: industrial pump control')

### Implicit intent — model must infer the right tool

In [ ]:
test("It's freezing in here",
     expect_tool='set_temperature',
     desc='Implicit: cold complaint → heating')

test("It's way too hot",
     expect_tool='set_temperature',
     desc='Implicit: hot complaint → cooling')

test("I can't see anything in the office",
     expect_tool='set_brightness',
     desc='Implicit: dark complaint → increase lights')

test('The lights are blinding me',
     expect_tool='set_brightness',
     desc='Implicit: bright complaint → dim lights')

test("I'm leaving, secure the house",
     expect_tool='lock_door',
     desc='Implicit: leaving → lock doors')

### Tool selection — model picks the right tool from the pool

In [ ]:
test('Send a critical alert about flooding in zone B',
     expect_tool='send_alert', expect_param={'level': 'critical'},
     desc='Selection: alert with severity level')

test('Temperature in the server room is 45°C',
     expect_tool='send_alert',
     desc='Selection: sensor reading → alert, not thermostat')

test('Start pump_7 at low flow, not high',
     expect_tool='start_pump', expect_param={'flow_rate': 'low'},
     desc='Selection: explicit param override')

### No tool needed — model should answer without calling tools

In [ ]:
test('What is a good room temperature?',
     expect_tool=False,
     desc='No tool: knowledge question')

test('Thanks!',
     expect_tool=False,
     desc='No tool: gratitude')

test('How does a thermostat work?',
     expect_tool=False,
     desc='No tool: explanation request')

### Precision — exact values must be preserved

In [ ]:
test('Set the bedroom lights to exactly 42%',
     expect_tool='set_brightness', expect_param={'level': 42},
     desc='Precision: exact brightness 42%')

test('Set temperature to 19.5, heating only',
     expect_tool='set_temperature', expect_param={'temperature': 19.5, 'mode': 'heat'},
     desc='Precision: decimal temp + specific mode')

test('Start pump_11 at medium',
     expect_tool='start_pump', expect_param={'pump_id': 'pump_11', 'flow_rate': 'medium'},
     desc='Precision: exact pump ID + flow rate')

### Test Summary

In [ ]:
# Run all tests again and tally results
import re

ALL_TESTS = [
    # (prompt, expect_tool, expect_param, desc)
    ('Turn on the kitchen lights', 'turn_on', {'device': 'kitchen lights'}, 'Direct: turn on'),
    ('Turn off the bedroom fan', 'turn_off', {'device': 'bedroom fan'}, 'Direct: turn off'),
    ('Lock the front door', 'lock_door', {'door': 'front door'}, 'Direct: lock'),
    ('Set the temperature to 22 degrees, cooling mode', 'set_temperature', {'temperature': 22, 'mode': 'cool'}, 'Direct: thermostat'),
    ('Dim the living room lights to 30%', 'set_brightness', {'level': 30}, 'Direct: brightness'),
    ('Set a timer for 15 minutes for the pasta', 'set_timer', {'duration': '15 minutes'}, 'Direct: timer'),
    ('Start pump_3 at high flow', 'start_pump', {'pump_id': 'pump_3', 'flow_rate': 'high'}, 'Direct: pump'),
    ("It's freezing in here", 'set_temperature', None, 'Implicit: cold'),
    ("It's way too hot", 'set_temperature', None, 'Implicit: hot'),
    ("I can't see anything in the office", 'set_brightness', None, 'Implicit: dark'),
    ('The lights are blinding me', 'set_brightness', None, 'Implicit: bright'),
    ("I'm leaving, secure the house", 'lock_door', None, 'Implicit: secure'),
    ('Send a critical alert about flooding in zone B', 'send_alert', {'level': 'critical'}, 'Select: alert'),
    ('Start pump_7 at low flow, not high', 'start_pump', {'flow_rate': 'low'}, 'Select: param override'),
    ('What is a good room temperature?', False, None, 'No tool: knowledge'),
    ('Thanks!', False, None, 'No tool: gratitude'),
    ('How does a thermostat work?', False, None, 'No tool: explain'),
    ('Set the bedroom lights to exactly 42%', 'set_brightness', {'level': 42}, 'Precision: 42%'),
    ('Set temperature to 19.5, heating only', 'set_temperature', {'temperature': 19.5, 'mode': 'heat'}, 'Precision: 19.5°C'),
    ('Start pump_11 at medium', 'start_pump', {'pump_id': 'pump_11', 'flow_rate': 'medium'}, 'Precision: pump_11'),
]

passed = 0
total = len(ALL_TESTS)
for prompt, expect_tool, expect_param, desc in ALL_TESTS:
    r = engine.chat_completion(
        [{'role': 'system', 'content': SYSTEM}, {'role': 'user', 'content': prompt}],
        tools=TOOLS
    )
    msg = r['choices'][0]['message']
    tc = msg.get('tool_calls', [None])[0] if msg.get('tool_calls') else None
    name = tc['function']['name'] if tc else None
    args = json.loads(tc['function']['arguments']) if tc else {}

    ok = True
    if expect_tool is False:
        ok = name is None
    elif expect_tool:
        ok = name == expect_tool
        if ok and expect_param:
            for k, v in expect_param.items():
                if str(args.get(k, '')).lower() != str(v).lower():
                    ok = False
    if ok:
        passed += 1

print(f'\n{"="*50}')
print(f'Tool Calling Score: {passed}/{total} ({passed/total*100:.0f}%)')
print(f'{"="*50}')
print(f'Trained for 10,000 steps. Adjust max_steps in config.py to train more or less.')

## 7. Benchmark Early Exit

In [ ]:
engine.benchmark()

## 8. Export C Runtime

Export model to `.goby` binary and compile native C runtime.
**50-200x faster** than Python on RPi.

In [ ]:
!python export_goby.py --checkpoint checkpoints/best_model.pt \
                       --tokenizer data/tokenizer.json \
                       --output goby.bin

import os
print(f'Model binary: {os.path.getsize("goby.bin")/1e6:.1f} MB')

In [ ]:
!make clean && make
!ls -lh goby

In [ ]:
!./goby goby.bin -b

## 9. Interactive C Runtime

Test tool-calling prompts through the C binary.

In [ ]:
import subprocess, shlex

def goby(prompt, n=128, t=0.7):
    r = subprocess.run(f'./goby goby.bin -p {shlex.quote(prompt)} -n {n} -t {t}',
                       shell=True, capture_output=True, text=True)
    print(f'You> {prompt}')
    print(f'Goby> {r.stdout}')
    if r.stderr: print(r.stderr.strip())
    print()

goby('Turn on the kitchen lights')
goby('Set temperature to 22, cooling')
goby('Lock all the doors')
goby('Start pump_5 at max flow')
goby('Send a critical alert about gas leak in zone A')

In [ ]:
#@title Try your own prompt { run: "auto" }
custom_prompt = 'Dim the office lights to 40 percent' #@param {type:"string"}
max_tokens = 128 #@param {type:"slider", min:16, max:512, step:16}
temperature = 0.7 #@param {type:"slider", min:0.1, max:1.5, step:0.1}

goby(custom_prompt, n=max_tokens, t=temperature)

In [ ]:
# Speed comparison: Python vs C on the same tool-calling prompt
import time

prompt = 'Turn on the bedroom lights'

t0 = time.time()
r = engine.chat_completion(
    [{'role': 'system', 'content': SYSTEM}, {'role': 'user', 'content': prompt}],
    tools=TOOLS, max_tokens=64
)
py_ms = (time.time() - t0) * 1000

t0 = time.time()
subprocess.run(f'./goby goby.bin -p {shlex.quote(prompt)} -n 64 -t 0.7',
               shell=True, capture_output=True, text=True)
c_ms = (time.time() - t0) * 1000

print(f'Python: {py_ms:.0f}ms')
print(f'C:      {c_ms:.0f}ms')
print(f'Speedup: {py_ms/c_ms:.1f}x')

## 10. Download

In [ ]:
import os
!cd /content && tar czf goby_llm.tar.gz \
    goby/goby.bin \
    goby/goby.c \
    goby/Makefile \
    goby/goby \
    goby/checkpoints/best_model.pt \
    goby/checkpoints/config.json \
    goby/data/tokenizer.json \
    goby/model.py \
    goby/config.py \
    goby/inference.py \
    goby/rpi_runner.py \
    goby/export_goby.py

sz = os.path.getsize('/content/goby_llm.tar.gz') / 1e6
print(f'Package: /content/goby_llm.tar.gz ({sz:.1f} MB)')

try:
    from google.colab import files
    files.download('/content/goby_llm.tar.gz')
except ImportError:
    print('Not in Colab — download manually.')

## 11. Deploy on Edge

**Three runtime options** (fastest to slowest):

### Option 1: C Runtime (fastest — recommended for RPi)
```bash
tar xzf goby_llm.tar.gz && cd goby

# Re-compile for RPi (if downloaded Colab x86 binary):
make clean && make

# Run:
./goby goby.bin -p 'Turn on the lights'    # single prompt
./goby goby.bin -i                          # interactive
./goby goby.bin -b                          # benchmark
```
Zero dependencies. ~51KB binary. mmap model loading. KV cache + early exit.

### Option 2: Python KV-cached runner (needs torch + tokenizers)
```bash
pip install torch tokenizers
python3 rpi_runner.py --serve --port 8000   # OpenAI-compatible server
```

### Option 3: Python naive (slowest, for debugging)
```bash
python3 inference.py --interactive
```